In [ ]:
#==============================================================================
# 0.import library
#==============================================================================

!pip install rdkit

import sys
import subprocess
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
from collections import Counter
import gc

# Check and install required packages
def ensure_package_installed(package_name, import_name=None):
    """Ensure a package is installed"""
    if import_name is None:
        import_name = package_name

    try:
        __import__(import_name)
        return True
    except ImportError:
        print(f"Installing {package_name}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
            return True
        except:
            return False

# Essential packages
essential_packages = [
    ('rdkit-pypi', 'rdkit'),
    ('scikit-learn', 'sklearn'),
    ('xgboost', 'xgboost'),
    ('torch', 'torch'),
    ('scipy', 'scipy'),
    ('shap', 'shap')
]

print("Checking required packages...")
for package, import_name in essential_packages:
    ensure_package_installed(package, import_name)

# Import required libraries
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    auc
)
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from scipy import stats
from scipy.optimize import minimize
import xgboost as xgb
import torch
import torch.nn as nn
import torch.optim as optim
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Descriptors3D, Lipinski, rdMolDescriptors
import shap

# Configure environment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RDLogger.DisableLog('rdApp.*')
print(f"Environment configured. Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 27.1 MB/s eta 0:00:00
Checking required packages...
Environment configured. Device: cpu


In [ ]:
#==============================================================================
# 1. Memory optimization
#==============================================================================

def optimize_memory_usage():
    """Clean up memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
#==============================================================================
# 2. Optimized GSK3 descriptors (8 features only)
#==============================================================================

def enhanced_gsk3_descriptors(mol):
    """
    Enhanced GSK3 descriptors - OPTIMIZED VERSION with 8 statistically significant features
    Based on ProLIF correlation analysis (p<0.05)

    Returns 8 features:
    - GSK002: HBond_Donors_Ratio (p=0.0419)
    - GSK004: Total_HBond_Potential (p=0.0180)
    - GSK007: Aromatic_Carbocycles (p=0.0237)
    - GSK008: Aromatic_Heterocycles (p=0.0074)
    - GSK015: Rotatable_Bonds (p=0.0101)
    - GSK026: Negative_Charges (p=0.0101)
    - GSK037: Basic_N_Ratio (p=0.0306)
    - GSK043: Aromatic_Ring_Fraction (p=0.0074)
    """
    if mol is None:
        return np.zeros(8)

    descriptors = []

    try:
        # Basic molecular properties
        heavy_atoms = Descriptors.HeavyAtomCount(mol)
        heavy_atoms_safe = max(1, heavy_atoms)

        # Hydrogen bonding properties
        hbond_donors = Lipinski.NumHDonors(mol)
        hbond_acceptors = Lipinski.NumHAcceptors(mol)

        # Ring information
        ring_info = mol.GetRingInfo()
        aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
        aromatic_carbocycles = rdMolDescriptors.CalcNumAromaticCarbocycles(mol)
        aromatic_heterocycles = rdMolDescriptors.CalcNumAromaticHeterocycles(mol)
        total_rings = ring_info.NumRings()
        total_rings_safe = max(1, total_rings)

        # Rotatable bonds
        rotatable_bonds = Descriptors.NumRotatableBonds(mol)

        # Charge-related features
        formal_charges = [atom.GetFormalCharge() for atom in mol.GetAtoms()]
        negative_charges = sum(1 for c in formal_charges if c < 0)

        # Calculate basic nitrogen count for GSK037
        basic_n_count = 0
        for atom in mol.GetAtoms():
            if atom.GetAtomicNum() == 7:  # Nitrogen
                if atom.GetTotalNumHs() > 0 and atom.GetTotalDegree() <= 3:
                    basic_n_count += 1

        # Calculate 8 features
        descriptors.append(hbond_donors / heavy_atoms_safe)  # GSK002
        descriptors.append((hbond_donors + hbond_acceptors) / heavy_atoms_safe)  # GSK004
        descriptors.append(aromatic_carbocycles)  # GSK007
        descriptors.append(aromatic_heterocycles)  # GSK008
        descriptors.append(rotatable_bonds)  # GSK015
        descriptors.append(negative_charges)  # GSK026
        descriptors.append(basic_n_count / heavy_atoms_safe)  # GSK037

        # GSK043: Aromatic_Ring_Fraction
        if total_rings > 0:
            descriptors.append(aromatic_rings / total_rings_safe)
        else:
            descriptors.append(0)

    except Exception as e:
        print(f"Error calculating GSK3 descriptors: {e}")
        descriptors = [0] * 8

    # Ensure exactly 8 features
    if len(descriptors) != 8:
        descriptors = [0] * 8

    # Replace NaN/Inf with 0
    descriptors = [0 if x is None or np.isnan(x) or np.isinf(x) else x
                   for x in descriptors]

    return np.array(descriptors)

In [ ]:
#==============================================================================
# 3. Essential 2D descriptors
#==============================================================================

def compute_essential_2d_descriptors(mol):
    """Calculate essential 2D molecular descriptors"""
    if mol is None:
        return [0] * 50

    result_descriptors = []
    try:
        # Basic properties
        result_descriptors.extend([
            Descriptors.MolWt(mol),
            Descriptors.HeavyAtomMolWt(mol),
            Descriptors.ExactMolWt(mol),
            Descriptors.NumValenceElectrons(mol),
            Descriptors.MaxPartialCharge(mol),
            Descriptors.MinPartialCharge(mol),
            Descriptors.MaxAbsPartialCharge(mol),
            Descriptors.MinAbsPartialCharge(mol)
        ])

        # Lipinski descriptors
        result_descriptors.extend([
            Lipinski.NumHDonors(mol),
            Lipinski.NumHAcceptors(mol),
            Lipinski.NumHeteroatoms(mol),
            Lipinski.NumRotatableBonds(mol),
            Lipinski.NumAliphaticRings(mol),
            Lipinski.NumAromaticRings(mol),
            Lipinski.NumSaturatedRings(mol),
            Lipinski.RingCount(mol)
        ])

        # Physicochemical properties
        result_descriptors.extend([
            Descriptors.MolLogP(mol),
            Descriptors.MolMR(mol),
            Descriptors.TPSA(mol),
            Descriptors.LabuteASA(mol)
        ])

        # Counts
        result_descriptors.extend([
            Descriptors.HeavyAtomCount(mol),
            Descriptors.NHOHCount(mol),
            Descriptors.NOCount(mol),
            Descriptors.NumAliphaticCarbocycles(mol),
            Descriptors.NumAliphaticHeterocycles(mol),
            Descriptors.NumAromaticCarbocycles(mol),
            Descriptors.NumAromaticHeterocycles(mol),
            Descriptors.NumSaturatedCarbocycles(mol),
            Descriptors.NumSaturatedHeterocycles(mol)
        ])

        # Graph descriptors
        result_descriptors.extend([
            Descriptors.BalabanJ(mol),
            Descriptors.BertzCT(mol),
            Descriptors.Chi0(mol),
            Descriptors.Chi0n(mol),
            Descriptors.Chi0v(mol),
            Descriptors.Chi1(mol),
            Descriptors.Chi1n(mol),
            Descriptors.Chi1v(mol),
            Descriptors.Chi2n(mol),
            Descriptors.Chi2v(mol),
            Descriptors.Chi3n(mol),
            Descriptors.Chi3v(mol),
            Descriptors.Chi4n(mol),
            Descriptors.Chi4v(mol),
            Descriptors.HallKierAlpha(mol),
            Descriptors.Kappa1(mol),
            Descriptors.Kappa2(mol),
            Descriptors.Kappa3(mol)
        ])

    except Exception as e:
        print(f"Error in 2D descriptors: {e}")
        result_descriptors = [0] * 50

    if len(result_descriptors) != 50:
        result_descriptors = [0] * 50

    return result_descriptors

In [ ]:
#==============================================================================
# 4. 3D descriptors
#==============================================================================

def generate_3d_conformation(mol):
    """Generate 3D conformation"""
    if mol is None:
        return None

    try:
        mol_3d = Chem.AddHs(mol)
        result = AllChem.EmbedMolecule(mol_3d, randomSeed=42, maxAttempts=100)

        if result == 0:
            AllChem.MMFFOptimizeMolecule(mol_3d, maxIters=500)
            return mol_3d
    except:
        pass

    return None

def calculate_3d_descriptors(mol):
    """Calculate 3D molecular descriptors"""
    if mol is None or mol.GetNumConformers() == 0:
        return [0] * 20

    descriptors_3d = []

    try:
        descriptors_3d.extend([
            Descriptors3D.Asphericity(mol),
            Descriptors3D.Eccentricity(mol),
            Descriptors3D.InertialShapeFactor(mol),
            Descriptors3D.NPR1(mol),
            Descriptors3D.NPR2(mol),
            Descriptors3D.PMI1(mol),
            Descriptors3D.PMI2(mol),
            Descriptors3D.PMI3(mol),
            Descriptors3D.RadiusOfGyration(mol),
            Descriptors3D.SpherocityIndex(mol)
        ])

        # Additional 3D descriptors
        descriptors_3d.extend([
            mol.GetConformer().GetPositions().mean(),
            mol.GetConformer().GetPositions().std(),
            mol.GetConformer().GetPositions().max(),
            mol.GetConformer().GetPositions().min(),
            mol.GetNumConformers()
        ])

        # Pad to 20 features
        while len(descriptors_3d) < 20:
            descriptors_3d.append(0)

    except Exception as e:
        descriptors_3d = [0] * 20

    return descriptors_3d[:20]

In [ ]:
#==============================================================================
# 5. Molecular fingerprints
#==============================================================================

def generate_molecular_fingerprint(mol, radius=3, nBits=1024):
    """Generate ECFP fingerprint"""
    if mol is None:
        return np.zeros(nBits)

    try:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
        return np.array(fp)
    except:
        return np.zeros(nBits)

In [ ]:
#==============================================================================
# 6. Feature preprocessing pipeline
#==============================================================================

class FeaturePreprocessingPipeline:
    """Feature preprocessing and scaling pipeline"""

    def __init__(self):
        self.scaler = RobustScaler()
        self.original_feature_dimension = None
        self.constant_feature_mask = None
        self.pipeline_fitted = False

    def fit(self, X, y=None):
        """Fit the preprocessing pipeline"""
        print(f"\nFitting preprocessing pipeline...")
        print(f"Input shape: {X.shape}")

        self.original_feature_dimension = X.shape[1]

        # Remove constant features
        feature_variances = np.var(X, axis=0)
        self.constant_feature_mask = feature_variances > 1e-8
        X_filtered = X[:, self.constant_feature_mask]

        n_removed = np.sum(~self.constant_feature_mask)
        print(f"Removed {n_removed} constant features")
        print(f"Remaining features: {X_filtered.shape[1]}")

        # Fit scaler
        self.scaler.fit(X_filtered)
        self.pipeline_fitted = True

        return self

    def transform(self, X):
        """Transform features"""
        if not self.pipeline_fitted:
            raise ValueError("Pipeline must be fitted first")

        # Handle dimension mismatch
        if X.shape[1] != self.original_feature_dimension:
            print(f"Warning: Dimension mismatch. Expected {self.original_feature_dimension}, got {X.shape[1]}")
            if X.shape[1] > self.original_feature_dimension:
                X = X[:, :self.original_feature_dimension]
            else:
                padding = np.zeros((X.shape[0], self.original_feature_dimension - X.shape[1]))
                X = np.hstack([X, padding])

        # Apply masks and scaling
        X_filtered = X[:, self.constant_feature_mask]
        X_scaled = self.scaler.transform(X_filtered)

        return X_scaled

    def fit_transform(self, X, y=None):
        """Fit and transform"""
        self.fit(X, y)
        return self.transform(X)

    def save(self, filepath):
        """Save pipeline"""
        pipeline_data = {
            'original_feature_dimension': self.original_feature_dimension,
            'scaler': self.scaler,
            'constant_feature_mask': self.constant_feature_mask,
            'pipeline_fitted': self.pipeline_fitted
        }
        joblib.dump(pipeline_data, filepath)
        print(f"Pipeline saved to {filepath}")

    @classmethod
    def load(cls, filepath):
        """Load pipeline"""
        pipeline_data = joblib.load(filepath)
        pipeline = cls()
        for key, value in pipeline_data.items():
            setattr(pipeline, key, value)
        return pipeline

In [ ]:
#==============================================================================
# 7. Enhanced Neural Network
#==============================================================================

class EnhancedGSK3NeuralNetwork(nn.Module):
    """Enhanced neural network for GSK3-beta activity prediction"""

    def __init__(self, input_size):
        super(EnhancedGSK3NeuralNetwork, self).__init__()

        hidden1_size = max(256, input_size // 2)
        hidden2_size = max(128, input_size // 4)
        hidden3_size = max(64, input_size // 8)

        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden1_size),
            nn.BatchNorm1d(hidden1_size),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden1_size, hidden2_size),
            nn.BatchNorm1d(hidden2_size),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(hidden2_size, hidden3_size),
            nn.BatchNorm1d(hidden3_size),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(hidden3_size, 1)
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            module.bias.data.fill_(0.01)

    def forward(self, x):
        return self.layers(x)

class NeuralNetworkWrapper:
    """
    Scikit-learn compatible wrapper for PyTorch neural network

    This wrapper provides fit() and predict() methods compatible with
    cross-validation and ensemble systems.
    """

    def __init__(self, input_size=None, epochs=200, learning_rate=0.001,
                 batch_size=32, device_param=None, validation_split=0.2):
        """
        Args:
            input_size: Number of input features (set during fit if None)
            epochs: Maximum number of training epochs
            learning_rate: Learning rate for Adam optimizer
            batch_size: Batch size for training
            device_param: torch device (cuda/cpu)
            validation_split: Fraction of data to use for validation
        """
        self.input_size = input_size
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.device_param = device_param if device_param else device
        self.validation_split = validation_split
        self.model = None
        self.is_fitted_ = False

    def fit(self, X, y):
        """
        Fit the neural network model

        Args:
            X: Training features (numpy array or tensor)
            y: Training targets (numpy array or tensor)

        Returns:
            self
        """
        # Convert to numpy if tensor
        if isinstance(X, torch.Tensor):
            X = X.cpu().numpy()
        if isinstance(y, torch.Tensor):
            y = y.cpu().numpy()

        # Set input size if not set
        if self.input_size is None:
            self.input_size = X.shape[1]

        # Create validation split
        n_samples = X.shape[0]
        n_val = max(1, int(n_samples * self.validation_split))

        # Shuffle indices
        indices = np.random.permutation(n_samples)
        train_idx = indices[n_val:]
        val_idx = indices[:n_val]

        X_train = X[train_idx]
        y_train = y[train_idx]
        X_val = X[val_idx] if n_val > 0 else X_train[:min(10, len(X_train))]
        y_val = y[val_idx] if n_val > 0 else y_train[:min(10, len(y_train))]

        # Initialize model
        self.model = EnhancedGSK3NeuralNetwork(self.input_size).to(self.device_param)
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=0.01)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)

        # Convert to tensors
        X_train_tensor = torch.FloatTensor(X_train).to(self.device_param)
        y_train_tensor = torch.FloatTensor(y_train.reshape(-1, 1)).to(self.device_param)
        X_val_tensor = torch.FloatTensor(X_val).to(self.device_param)
        y_val_tensor = torch.FloatTensor(y_val.reshape(-1, 1)).to(self.device_param)

        # Training loop
        best_val_loss = float('inf')
        patience_counter = 0
        max_patience = 25

        for epoch in range(self.epochs):
            # Training phase
            self.model.train()
            optimizer.zero_grad()

            # Forward pass
            outputs = self.model(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()

            # Validation phase
            self.model.eval()
            with torch.no_grad():
                val_outputs = self.model(X_val_tensor)
                val_loss = criterion(val_outputs, y_val_tensor)

            scheduler.step(val_loss)

            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= max_patience:
                    break

        self.is_fitted_ = True
        return self

    def predict(self, X):
        """
        Make predictions

        Args:
            X: Input features (numpy array or tensor)

        Returns:
            predictions: numpy array of predictions
        """
        if not self.is_fitted_:
            raise ValueError("Model must be fitted before making predictions")

        # Convert to numpy if tensor
        if isinstance(X, torch.Tensor):
            X = X.cpu().numpy()

        # Set model to evaluation mode
        self.model.eval()

        # Convert to tensor
        X_tensor = torch.FloatTensor(X).to(self.device_param)

        # Make predictions
        with torch.no_grad():
            predictions = self.model(X_tensor).cpu().numpy().flatten()

        return predictions

def train_neural_network(X_train, y_train, X_val, y_val, input_size, epochs=200):
    """Train neural network"""
    model = EnhancedGSK3NeuralNetwork(input_size).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5)

    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_tensor = torch.FloatTensor(y_train.reshape(-1, 1)).to(device)
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_tensor = torch.FloatTensor(y_val.reshape(-1, 1)).to(device)

    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_tensor)
            val_loss = criterion(val_outputs, y_val_tensor)
            val_losses.append(val_loss.item())

        scheduler.step(val_loss)

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 25:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 50 == 0:
            print(f"Epoch {epoch}: Train Loss = {loss.item():.6f}, Val Loss = {val_loss.item():.6f}")

    return model, train_losses, val_losses

In [ ]:
#==============================================================================
# 8. Ensemble System
#==============================================================================

class EnhancedEnsembleSystem:
    """Enhanced ensemble system with safety checks"""

    def __init__(self, feature_pipeline):
        self.feature_pipeline = feature_pipeline
        self.models = {}
        self.weights = {}
        self.is_fitted = False

    def fit(self, X_train, y_train, X_val, y_val):
        """Fit ensemble models with NaN safety checks"""
        print("\nTraining ensemble models...")

        print("Validating input data...")
        if np.isnan(X_train).any():
            print("WARNING: NaN in X_train. Replacing with 0...")
            X_train = np.nan_to_num(X_train, nan=0.0)

        if np.isnan(X_val).any():
            print("WARNING: NaN in X_val. Replacing with 0...")
            X_val = np.nan_to_num(X_val, nan=0.0)

        if np.isnan(y_train).any():
            print("ERROR: NaN found in y_train!")
            raise ValueError("y_train contains NaN values")

        if np.isnan(y_val).any():
            print("ERROR: NaN found in y_val!")
            raise ValueError("y_val contains NaN values")

        print("Input data validation passed!")

        # Model configurations
        model_configs = {
            'random_forest': RandomForestRegressor(
                n_estimators=200, max_depth=15, min_samples_split=5,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            ),
            'xgboost': xgb.XGBRegressor(
                n_estimators=200, max_depth=8, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, random_state=42
            ),
            'elastic_net': ElasticNet(
                alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=5000
            ),
            'svr': SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.1),
            'gradient_boosting': GradientBoostingRegressor(
                n_estimators=200, max_depth=8, learning_rate=0.1,
                subsample=0.8, random_state=42
            )
        }

        # Train models and calculate scores
        predictions = {}
        scores = {}

        for name, model in model_configs.items():
            try:
                print(f"Training {name}...")
                model.fit(X_train, y_train)
                val_pred = model.predict(X_val)

                if np.isnan(val_pred).any():
                    print(f"  WARNING: {name} produced NaN predictions. Skipping...")
                    continue

                score = r2_score(y_val, val_pred)

                self.models[name] = model
                predictions[name] = val_pred
                scores[name] = max(0.01, score)

                print(f"  {name} RÂ² = {score:.4f}")
            except Exception as e:
                print(f"  Error training {name}: {e}")

        # Train neural network using wrapper
        try:
            print("Training neural network...")

            nn_wrapper = NeuralNetworkWrapper(
                input_size=X_train.shape[1],
                epochs=200,
                learning_rate=0.001,
                device_param=device,
                validation_split=0.15
            )

            # Combine train and val for internal split
            X_combined = np.vstack([X_train, X_val])
            y_combined = np.hstack([y_train, y_val])

            nn_wrapper.fit(X_combined, y_combined)
            nn_pred = nn_wrapper.predict(X_val)

            if not np.isnan(nn_pred).any():
                nn_score = r2_score(y_val, nn_pred)

                self.models['neural_network'] = nn_wrapper
                predictions['neural_network'] = nn_pred
                scores['neural_network'] = max(0.01, nn_score)

                print(f"  Neural network R² = {nn_score:.4f}")
            else:
                print(f"  WARNING: Neural network produced NaN predictions. Skipping...")

        except Exception as e:
            print(f"  Error training neural network: {e}")
            import traceback
            traceback.print_exc()

        if len(predictions) == 0:
            raise ValueError("No models were successfully trained!")

        print(f"\nSuccessfully trained {len(predictions)} models")

        # Optimize ensemble weights
        self._optimize_weights(predictions, scores, y_val)
        self.is_fitted = True

        print(f"\nFinal ensemble weights: {self.weights}")

    def _optimize_weights(self, predictions, scores, y_true):
        """Optimize ensemble weights"""
        model_names = list(predictions.keys())
        pred_matrix = np.column_stack([predictions[name] for name in model_names])

        def objective(weights):
            weights = np.abs(weights)
            weights = weights / np.sum(weights)
            ensemble_pred = np.dot(pred_matrix, weights)
            return -r2_score(y_true, ensemble_pred)

        initial_weights = np.array([max(0.01, scores[name]) for name in model_names])
        initial_weights = initial_weights / np.sum(initial_weights)

        try:
            result = minimize(objective, initial_weights, method='SLSQP',
                            bounds=[(0.01, 1.0)] * len(model_names))
            optimized_weights = result.x
            optimized_weights = optimized_weights / np.sum(optimized_weights)
        except:
            optimized_weights = initial_weights

        self.weights = {name: weight for name, weight in zip(model_names, optimized_weights)}

    def predict(self, X):
        """Make predictions"""
        if not self.is_fitted:
            raise ValueError("Ensemble must be fitted first")

        if np.isnan(X).any():
            print("WARNING: NaN in prediction input. Replacing with 0...")
            X = np.nan_to_num(X, nan=0.0)

        predictions = {}

        for name, model in self.models.items():
            try:
                # All models now have predict() method (including neural network wrapper)
                pred = model.predict(X)

                if not np.isnan(pred).any():
                    predictions[name] = pred
                else:
                    print(f"WARNING: {name} produced NaN predictions during prediction")

            except Exception as e:
                print(f"Error predicting with {name}: {e}")

        # Weighted ensemble
        ensemble_pred = np.zeros(X.shape[0])
        total_weight = 0

        for name, pred in predictions.items():
            if name in self.weights:
                weight = self.weights[name]
                ensemble_pred += weight * pred
                total_weight += weight

        if total_weight > 0:
            ensemble_pred = ensemble_pred / total_weight

        return ensemble_pred

In [ ]:
#==============================================================================
# 9. Activity classification
#==============================================================================

def classify_activity(pchembl_values):
    """
    Classify compounds by activity
    Returns both multi-class and binary labels
    """
    multi_class = []
    binary = []

    for value in pchembl_values:
        if value < 5.5:
            multi_class.append('inactive')
            binary.append(0)  # inactive
        elif value < 7.5:
            multi_class.append('moderately_active')
            binary.append(1)  # active
        else:
            multi_class.append('highly_active')
            binary.append(1)  # active

    return multi_class, np.array(binary)


def classify_activity_multiclass(pchembl_values):
    """
    Classify into three classes with numeric labels
    0: inactive, 1: moderately_active, 2: highly_active
    """
    labels = []
    for value in pchembl_values:
        if value < 5.5:
            labels.append(0)  # inactive
        elif value < 7.5:
            labels.append(1)  # moderately_active
        else:
            labels.append(2)  # highly_active
    return np.array(labels)

# ==================================================================================
# AUC Metrics Calculation
# ==================================================================================

def calculate_auc_metrics(y_true, y_pred, mode='binary'):
    """
    Calculate AUC-ROC and AUC-PR metrics

    Args:
        y_true: True pChEMBL values
        y_pred: Predicted pChEMBL values
        mode: 'binary' or 'multiclass'

    Returns:
        dict with AUC metrics
    """
    results = {}

    if mode == 'binary':
        # Binary classification: active (>=5.5) vs inactive (<5.5)
        y_true_binary = (y_true >= 5.5).astype(int)
        y_pred_binary = (y_pred >= 5.5).astype(float)

        try:
            # ROC-AUC
            roc_auc = roc_auc_score(y_true_binary, y_pred)
            results['roc_auc_binary'] = roc_auc

            # PR-AUC
            pr_auc = average_precision_score(y_true_binary, y_pred)
            results['pr_auc_binary'] = pr_auc

        except Exception as e:
            print(f"Warning: Binary AUC calculation failed: {e}")
            results['roc_auc_binary'] = 0.0
            results['pr_auc_binary'] = 0.0

    elif mode == 'multiclass':
        # Multi-class: inactive, moderately_active, highly_active
        y_true_multi = classify_activity_multiclass(y_true)
        y_pred_multi = classify_activity_multiclass(y_pred)

        try:
            # One-vs-Rest ROC-AUC
            y_true_bin = label_binarize(y_true_multi, classes=[0, 1, 2])
            y_pred_bin = label_binarize(y_pred_multi, classes=[0, 1, 2])

            roc_auc_ovr = roc_auc_score(y_true_bin, y_pred_bin,
                                        average='macro', multi_class='ovr')
            results['roc_auc_multiclass'] = roc_auc_ovr

        except Exception as e:
            print(f"Warning: Multiclass AUC calculation failed: {e}")
            results['roc_auc_multiclass'] = 0.0

    return results


def plot_auc_curves(y_true, y_pred, save_path, title_prefix=''):
    """
    Plot ROC and PR curves
    """
    # Binary classification
    y_true_binary = (y_true >= 5.5).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true_binary, y_pred)
    roc_auc = auc(fpr, tpr)

    axes[0].plot(fpr, tpr, color='darkorange', lw=2,
                 label=f'ROC curve (AUC = {roc_auc:.3f})')
    axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    axes[0].set_title(f'{title_prefix}ROC Curve', fontsize=13, fontweight='bold')
    axes[0].legend(loc="lower right", fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # PR Curve
    precision, recall, _ = precision_recall_curve(y_true_binary, y_pred)
    pr_auc = auc(recall, precision)

    axes[1].plot(recall, precision, color='blue', lw=2,
                 label=f'PR curve (AUC = {pr_auc:.3f})')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('Recall', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Precision', fontsize=12, fontweight='bold')
    axes[1].set_title(f'{title_prefix}Precision-Recall Curve',
                     fontsize=13, fontweight='bold')
    axes[1].legend(loc="lower left", fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    return roc_auc, pr_auc


def enhanced_cross_validation_with_auc(X, y, ensemble_system, n_folds=5):
    """
    Enhanced cross-validation with AUC-ROC and AUC-PR metrics

    Args:
        X: Feature matrix
        y: Target values
        ensemble_system: Trained ensemble system
        n_folds: Number of folds (5 or 10)

    Returns:
        dict with all metrics
    """
    print(f"\n{'='*70}")
    print(f"{n_folds}-FOLD CROSS-VALIDATION WITH AUC METRICS")
    print(f"{'='*70}")

    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)

    fold_r2 = []
    fold_rmse = []
    fold_mae = []
    fold_roc_auc = []
    fold_pr_auc = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Train models
        for name, model in ensemble_system.models.items():
            model.fit(X_train, y_train)

        # Predict
        y_pred = ensemble_system.predict(X_val)

        # Regression metrics
        r2 = r2_score(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)

        fold_r2.append(r2)
        fold_rmse.append(rmse)
        fold_mae.append(mae)

        # AUC metrics
        auc_metrics = calculate_auc_metrics(y_val, y_pred, mode='binary')
        fold_roc_auc.append(auc_metrics.get('roc_auc_binary', 0.0))
        fold_pr_auc.append(auc_metrics.get('pr_auc_binary', 0.0))

        print(f"Fold {fold:2d}: RÂ²={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, "
              f"ROC-AUC={auc_metrics.get('roc_auc_binary', 0.0):.4f}, "
              f"PR-AUC={auc_metrics.get('pr_auc_binary', 0.0):.4f}")

    results = {
        'n_folds': n_folds,
        'r2_mean': np.mean(fold_r2),
        'r2_std': np.std(fold_r2),
        'rmse_mean': np.mean(fold_rmse),
        'rmse_std': np.std(fold_rmse),
        'mae_mean': np.mean(fold_mae),
        'mae_std': np.std(fold_mae),
        'roc_auc_mean': np.mean(fold_roc_auc),
        'roc_auc_std': np.std(fold_roc_auc),
        'pr_auc_mean': np.mean(fold_pr_auc),
        'pr_auc_std': np.std(fold_pr_auc)
    }

    print(f"\n{'='*70}")
    print(f"{n_folds}-FOLD CV SUMMARY")
    print(f"{'='*70}")
    print(f"RÂ²:       {results['r2_mean']:.4f} Â± {results['r2_std']:.4f}")
    print(f"RMSE:     {results['rmse_mean']:.4f} Â± {results['rmse_std']:.4f}")
    print(f"MAE:      {results['mae_mean']:.4f} Â± {results['mae_std']:.4f}")
    print(f"ROC-AUC:  {results['roc_auc_mean']:.4f} Â± {results['roc_auc_std']:.4f}")
    print(f"PR-AUC:   {results['pr_auc_mean']:.4f} Â± {results['pr_auc_std']:.4f}")
    print(f"{'='*70}\n")

    return results

In [ ]:
#==============================================================================
# 10. SHAP analysis
#==============================================================================

def perform_shap_analysis(model, X_test, feature_names, output_dir):
    """Perform SHAP analysis with safety checks"""
    try:
        print("\nPerforming SHAP analysis...")
        shap_dir = f"{output_dir}/shap_analysis"
        os.makedirs(shap_dir, exist_ok=True)

        if len(feature_names) != X_test.shape[1]:
            print(f"WARNING: Feature names ({len(feature_names)}) != X_test features ({X_test.shape[1]})")
            if len(feature_names) > X_test.shape[1]:
                feature_names = feature_names[:X_test.shape[1]]
            else:
                for i in range(len(feature_names), X_test.shape[1]):
                    feature_names.append(f"Feature_{i:04d}")
            print(f"Adjusted feature_names length to {len(feature_names)}")

        # Limit samples for SHAP
        n_samples = min(100, X_test.shape[0])
        X_shap = X_test[:n_samples]

        print(f"Computing SHAP values for {n_samples} samples...")

        # Create explainer
        if isinstance(model, RandomForestRegressor):
            explainer = shap.TreeExplainer(model)
        else:
            explainer = shap.KernelExplainer(model.predict, X_shap[:50])

        # Calculate SHAP values
        shap_values = explainer.shap_values(X_shap)

        if isinstance(shap_values, list):
            shap_values = shap_values[0]

        print(f"SHAP values shape: {shap_values.shape}")

        # Summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                         show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(f"{shap_dir}/shap_summary.png", dpi=300, bbox_inches='tight')
        plt.close()

        # Feature importance
        feature_importance = pd.DataFrame({
            'feature': feature_names[:shap_values.shape[1]],
            'importance': np.abs(shap_values).mean(axis=0)
        }).sort_values('importance', ascending=False)

        feature_importance.to_csv(f"{shap_dir}/feature_importance.csv", index=False)

        print(f"SHAP analysis completed. Results saved to {shap_dir}")
        return feature_importance

    except Exception as e:
        print(f"Error in SHAP analysis: {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
#==============================================================================
# 11.mRMR Feature Selection with ProLIF Protection
#==============================================================================

def mrmr_feature_selection_with_prolif(X, y, feature_names, n_selected=400,
                                       prolif_indices=None):
    """
    mRMR (minimum Redundancy Maximum Relevance) Feature Selection
    with ProLIF feature protection

    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target values (n_samples,)
        feature_names: List of feature names
        n_selected: Total number of features to select (default: 400)
        prolif_indices: Indices of ProLIF features to protect (must include)

    Returns:
        selected_indices: Array of selected feature indices
        selected_names: List of selected feature names
    """
    print("\n" + "="*70)
    print("mRMR FEATURE SELECTION WITH PROLIF PROTECTION")
    print("="*70)

    if prolif_indices is None:
        prolif_indices = []

    try:
        from sklearn.feature_selection import mutual_info_regression

        # Separate ProLIF and non-ProLIF features
        non_prolif_mask = np.ones(len(feature_names), dtype=bool)
        if len(prolif_indices) > 0:
            non_prolif_mask[prolif_indices] = False

        non_prolif_features = [feature_names[i] for i in range(len(feature_names))
                               if non_prolif_mask[i]]

        print(f"\nTotal features: {len(feature_names)}")
        print(f"ProLIF protected: {len(prolif_indices)}")
        print(f"Available for selection: {len(non_prolif_features)}")

        # Calculate number to select
        n_to_select = min(n_selected - len(prolif_indices), len(non_prolif_features))

        if n_to_select > 0:
            print(f"\nSelecting {n_to_select} features via mRMR algorithm...")

            # Extract non-ProLIF features
            X_non_prolif = X[:, non_prolif_mask]

            # Calculate mutual information (Relevance)
            print("  Computing mutual information with target...")
            mi_scores = mutual_info_regression(X_non_prolif, y, random_state=42)

            # Calculate correlation matrix (for Redundancy)
            print("  Computing feature correlations...")
            correlations = np.corrcoef(X_non_prolif.T)

            # mRMR greedy selection
            print("  Performing greedy mRMR selection...")
            selected_indices_non_prolif = []
            remaining_indices = list(range(len(non_prolif_features)))

            for iteration in range(n_to_select):
                if not remaining_indices:
                    break

                scores = []
                for idx in remaining_indices:
                    # Relevance: MI with target
                    relevance = mi_scores[idx]

                    # Redundancy: average correlation with already selected features
                    if len(selected_indices_non_prolif) > 0:
                        redundancy = np.mean([abs(correlations[idx, sel_idx])
                                            for sel_idx in selected_indices_non_prolif])
                    else:
                        redundancy = 0

                    # mRMR score
                    mrmr_score = relevance - redundancy
                    scores.append((idx, mrmr_score))

                # Select best feature
                best_idx, best_score = max(scores, key=lambda x: x[1])
                selected_indices_non_prolif.append(best_idx)
                remaining_indices.remove(best_idx)

                # Progress indicator
                if (iteration + 1) % 50 == 0:
                    print(f"    Selected {iteration + 1}/{n_to_select} features...")

            # Convert to original indices
            non_prolif_indices_array = np.where(non_prolif_mask)[0]
            selected_from_mrmr = non_prolif_indices_array[selected_indices_non_prolif]
        else:
            selected_from_mrmr = []

        # Combine ProLIF + mRMR selected features
        all_selected = list(prolif_indices) + list(selected_from_mrmr)
        selected_names = [feature_names[i] for i in all_selected]

        print(f"\n" + "="*70)
        print("FEATURE SELECTION SUMMARY")
        print("="*70)
        print(f"ProLIF protected: {len(prolif_indices)}")
        print(f"mRMR selected: {len(selected_from_mrmr)}")
        print(f"Total selected: {len(all_selected)}")
        print(f"Selection rate: {len(all_selected)/len(feature_names)*100:.1f}%")

        return np.array(all_selected), selected_names

    except Exception as e:
        print(f"\nmRMR selection failed: {e}")
        print("Using correlation-based selection as fallback...")

        # Fallback: F-score based selection
        from sklearn.feature_selection import f_regression
        f_scores, _ = f_regression(X, y)

        # Mask ProLIF features
        non_prolif_mask = np.ones(len(feature_names), dtype=bool)
        if len(prolif_indices) > 0:
            non_prolif_mask[prolif_indices] = False

        f_scores_masked = f_scores.copy()
        f_scores_masked[~non_prolif_mask] = -np.inf

        n_to_select = min(n_selected - len(prolif_indices),
                         np.sum(non_prolif_mask))
        selected_from_f = np.argsort(f_scores_masked)[-n_to_select:]

        all_selected = list(prolif_indices) + list(selected_from_f)
        selected_names = [feature_names[i] for i in all_selected]

        print(f"\nFallback selection completed: {len(all_selected)} features")

        return np.array(all_selected), selected_names

In [ ]:
#==============================================================================
# 12.Ablation Study for Feature Group Importance
#==============================================================================

def perform_ablation_study(X, y, feature_groups, feature_names, cv_folds=5,
                           output_dir='gsk3_results'):
    """
    Ablation Study: Quantify feature group importance by systematic removal

    Args:
        X: Feature matrix (n_samples, n_features)
        y: Target values (n_samples,)
        feature_groups: Dict of {'group_name': [indices]}
        feature_names: List of feature names
        cv_folds: Number of cross-validation folds (default: 5)
        output_dir: Directory to save results

    Returns:
        ablation_results: DataFrame with ablation study results
    """
    print("\n" + "="*70)
    print("ABLATION STUDY - FEATURE GROUP IMPORTANCE ANALYSIS")
    print("="*70)

    ablation_dir = f"{output_dir}/ablation"
    os.makedirs(ablation_dir, exist_ok=True)

    results = []

    # Baseline: All features
    print("\nExperiment: All Features")
    print(f"  Number of features: {X.shape[1]}")

    model = GradientBoostingRegressor(
        n_estimators=100, random_state=42,
        max_depth=6, learning_rate=0.1
    )

    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
    fold_scores = []
    fold_rmses = []

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        r2 = r2_score(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        fold_scores.append(r2)
        fold_rmses.append(rmse)

        print(f"  Fold {fold}: RÂ² = {r2:.4f}, RMSE = {rmse:.4f}")

    baseline_r2 = np.mean(fold_scores)
    baseline_rmse = np.mean(fold_rmses)

    results.append({
        'experiment': 'All Features',
        'r2_mean': baseline_r2,
        'r2_std': np.std(fold_scores),
        'rmse_mean': baseline_rmse,
        'rmse_std': np.std(fold_rmses),
        'r2_decrease': 0.0,
        'n_features': X.shape[1]
    })

    print(f"  Average: RÂ² = {baseline_r2:.4f} Â± {np.std(fold_scores):.4f}, "
          f"RMSE = {baseline_rmse:.4f} Â± {np.std(fold_rmses):.4f}")

    # Ablation experiments: Remove each group
    for group_name, group_indices in feature_groups.items():
        print(f"\n{'='*70}")
        print(f"Experiment: No {group_name}")
        print(f"{'='*70}")
        print(f"  Excluded features: {len(group_indices)}")

        # Create mask to exclude this group
        keep_mask = np.ones(X.shape[1], dtype=bool)
        keep_mask[group_indices] = False
        X_reduced = X[:, keep_mask]

        print(f"  Remaining features: {X_reduced.shape[1]}")

        fold_scores = []
        fold_rmses = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(X_reduced), 1):
            X_train, X_val = X_reduced[train_idx], X_reduced[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = GradientBoostingRegressor(
                n_estimators=100, random_state=42,
                max_depth=6, learning_rate=0.1
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            r2 = r2_score(y_val, y_pred)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))

            fold_scores.append(r2)
            fold_rmses.append(rmse)

            print(f"  Fold {fold}: RÂ² = {r2:.4f}, RMSE = {rmse:.4f}")

        mean_r2 = np.mean(fold_scores)
        mean_rmse = np.mean(fold_rmses)
        r2_decrease = baseline_r2 - mean_r2

        results.append({
            'experiment': f'No {group_name}',
            'r2_mean': mean_r2,
            'r2_std': np.std(fold_scores),
            'rmse_mean': mean_rmse,
            'rmse_std': np.std(fold_rmses),
            'r2_decrease': r2_decrease,
            'n_features': X_reduced.shape[1]
        })

        print(f"  Average: RÂ² = {mean_r2:.4f} Â± {np.std(fold_scores):.4f}, "
              f"RMSE = {mean_rmse:.4f} Â± {np.std(fold_rmses):.4f}")
        print(f"  RÂ² Decrease: {r2_decrease:.4f}")

    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('r2_decrease', ascending=False)

    # Save results
    results_df.to_csv(f"{ablation_dir}/ablation_study_results.csv", index=False)

    # Print summary
    print("\n" + "="*70)
    print("ABLATION STUDY RESULTS SUMMARY")
    print("="*70)
    print(results_df.to_string(index=False))

    # Print importance ranking
    print("\n" + "="*70)
    print("FEATURE GROUP IMPORTANCE RANKING (by RÂ² decrease)")
    print("="*70)
    for i, row in enumerate(results_df[results_df['experiment'] != 'All Features'].iterrows(), 1):
        _, data = row
        print(f"  {i}. {data['experiment']}: RÂ² decrease = {data['r2_decrease']:.4f}")

    # Visualization
    print("\nCreating visualizations...")

    # Bar plot of RÂ² decrease
    plt.figure(figsize=(10, 6))
    ablation_plot_data = results_df[results_df['experiment'] != 'All Features']
    colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.9, len(ablation_plot_data)))

    bars = plt.barh(ablation_plot_data['experiment'],
                    ablation_plot_data['r2_decrease'],
                    color=colors)

    plt.xlabel('RÂ² Decrease (Importance)', fontsize=12, fontweight='bold')
    plt.ylabel('Ablation Experiment', fontsize=12, fontweight='bold')
    plt.title('Feature Group Importance (Ablation Study)',
              fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3, linestyle='--')

    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, ablation_plot_data['r2_decrease'])):
        plt.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig(f"{ablation_dir}/feature_group_importance.png",
                dpi=300, bbox_inches='tight')
    plt.close()

    # Performance comparison plot
    plt.figure(figsize=(12, 6))
    x_pos = np.arange(len(results_df))

    plt.subplot(1, 2, 1)
    plt.bar(x_pos, results_df['r2_mean'],
            yerr=results_df['r2_std'], capsize=5, alpha=0.7)
    plt.xlabel('Experiment', fontsize=11)
    plt.ylabel('RÂ² Score', fontsize=11)
    plt.title('RÂ² Performance Comparison', fontsize=12, fontweight='bold')
    plt.xticks(x_pos, results_df['experiment'], rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.bar(x_pos, results_df['rmse_mean'],
            yerr=results_df['rmse_std'], capsize=5, alpha=0.7, color='coral')
    plt.xlabel('Experiment', fontsize=11)
    plt.ylabel('RMSE', fontsize=11)
    plt.title('RMSE Performance Comparison', fontsize=12, fontweight='bold')
    plt.xticks(x_pos, results_df['experiment'], rotation=45, ha='right')
    plt.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{ablation_dir}/performance_comparison.png",
                dpi=300, bbox_inches='tight')
    plt.close()

    print(f"\nâœ“ Ablation study results saved to {ablation_dir}/")
    print(f"  - ablation_study_results.csv")
    print(f"  - feature_group_importance.png")
    print(f"  - performance_comparison.png")

    return results_df

def evaluate_plip_only_performance(X_plip, y, cv_folds=5, output_dir='gsk3_results'):
    """
    Evaluate model performance using ONLY ProLIF interaction features

    Args:
        X_plip: ProLIF feature matrix (8 features)
        y: Target values
        cv_folds: Number of cross-validation folds
        output_dir: Output directory

    Returns:
        dict with performance metrics
    """
    print("\n" + "="*70)
    print("PLIP (ProLIF) ONLY FEATURE PERFORMANCE EVALUATION")
    print("="*70)
    print(f"Using ONLY {X_plip.shape[1]} ProLIF-validated descriptors")
    print("="*70 + "\n")

    plip_dir = f"{output_dir}/plip_only"
    os.makedirs(plip_dir, exist_ok=True)

    results = {
        '5-fold': {},
        '10-fold': {},
        'test': {}
    }

    # Split data
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X_plip, y, test_size=0.15, random_state=42
    )

    # Train-validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.15/0.85, random_state=42
    )

    print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

    # Create simple ensemble with PLIP features only
    models = {
        'random_forest': RandomForestRegressor(n_estimators=100, max_depth=10,
                                              random_state=42),
        'gradient_boosting': GradientBoostingRegressor(n_estimators=100,
                                                       max_depth=6, random_state=42),
        'elastic_net': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)
    }

    # ========================================
    # 5-Fold Cross-Validation
    # ========================================
    print("\n5-FOLD CROSS-VALIDATION")
    print("-" * 70)

    kfold_5 = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_metrics_5 = {
        'r2': [], 'rmse': [], 'mae': [],
        'roc_auc': [], 'pr_auc': []
    }

    for fold, (train_idx, val_idx) in enumerate(kfold_5.split(X_train_val), 1):
        X_tr, X_vl = X_train_val[train_idx], X_train_val[val_idx]
        y_tr, y_vl = y_train_val[train_idx], y_train_val[val_idx]

        # Train all models
        fold_preds = []
        for name, model in models.items():
            model.fit(X_tr, y_tr)
            fold_preds.append(model.predict(X_vl))

        # Simple averaging ensemble
        y_pred = np.mean(fold_preds, axis=0)

        # Metrics
        r2 = r2_score(y_vl, y_pred)
        rmse = np.sqrt(mean_squared_error(y_vl, y_pred))
        mae = mean_absolute_error(y_vl, y_pred)

        auc_metrics = calculate_auc_metrics(y_vl, y_pred, mode='binary')

        fold_metrics_5['r2'].append(r2)
        fold_metrics_5['rmse'].append(rmse)
        fold_metrics_5['mae'].append(mae)
        fold_metrics_5['roc_auc'].append(auc_metrics.get('roc_auc_binary', 0.0))
        fold_metrics_5['pr_auc'].append(auc_metrics.get('pr_auc_binary', 0.0))

        print(f"Fold {fold}: RÂ²={r2:.4f}, RMSE={rmse:.4f}, "
              f"ROC-AUC={auc_metrics.get('roc_auc_binary', 0.0):.4f}")

    results['5-fold'] = {
        'r2_mean': np.mean(fold_metrics_5['r2']),
        'r2_std': np.std(fold_metrics_5['r2']),
        'rmse_mean': np.mean(fold_metrics_5['rmse']),
        'rmse_std': np.std(fold_metrics_5['rmse']),
        'mae_mean': np.mean(fold_metrics_5['mae']),
        'mae_std': np.std(fold_metrics_5['mae']),
        'roc_auc_mean': np.mean(fold_metrics_5['roc_auc']),
        'roc_auc_std': np.std(fold_metrics_5['roc_auc']),
        'pr_auc_mean': np.mean(fold_metrics_5['pr_auc']),
        'pr_auc_std': np.std(fold_metrics_5['pr_auc'])
    }

    print(f"\n5-Fold CV Average:")
    print(f"  RÂ²:       {results['5-fold']['r2_mean']:.4f} Â± {results['5-fold']['r2_std']:.4f}")
    print(f"  RMSE:     {results['5-fold']['rmse_mean']:.4f} Â± {results['5-fold']['rmse_std']:.4f}")
    print(f"  ROC-AUC:  {results['5-fold']['roc_auc_mean']:.4f} Â± {results['5-fold']['roc_auc_std']:.4f}")
    print(f"  PR-AUC:   {results['5-fold']['pr_auc_mean']:.4f} Â± {results['5-fold']['pr_auc_std']:.4f}")

    # ========================================
    # 10-Fold Cross-Validation
    # ========================================
    print("\n10-FOLD CROSS-VALIDATION")
    print("-" * 70)

    kfold_10 = KFold(n_splits=10, shuffle=True, random_state=42)
    fold_metrics_10 = {
        'r2': [], 'rmse': [], 'mae': [],
        'roc_auc': [], 'pr_auc': []
    }

    for fold, (train_idx, val_idx) in enumerate(kfold_10.split(X_train_val), 1):
        X_tr, X_vl = X_train_val[train_idx], X_train_val[val_idx]
        y_tr, y_vl = y_train_val[train_idx], y_train_val[val_idx]

        # Train all models
        fold_preds = []
        for name, model in models.items():
            model.fit(X_tr, y_tr)
            fold_preds.append(model.predict(X_vl))

        # Simple averaging ensemble
        y_pred = np.mean(fold_preds, axis=0)

        # Metrics
        r2 = r2_score(y_vl, y_pred)
        rmse = np.sqrt(mean_squared_error(y_vl, y_pred))
        mae = mean_absolute_error(y_vl, y_pred)

        auc_metrics = calculate_auc_metrics(y_vl, y_pred, mode='binary')

        fold_metrics_10['r2'].append(r2)
        fold_metrics_10['rmse'].append(rmse)
        fold_metrics_10['mae'].append(mae)
        fold_metrics_10['roc_auc'].append(auc_metrics.get('roc_auc_binary', 0.0))
        fold_metrics_10['pr_auc'].append(auc_metrics.get('pr_auc_binary', 0.0))

        if fold % 2 == 0:  # Print every 2 folds
            print(f"Fold {fold}: RÂ²={r2:.4f}, RMSE={rmse:.4f}, "
                  f"ROC-AUC={auc_metrics.get('roc_auc_binary', 0.0):.4f}")

    results['10-fold'] = {
        'r2_mean': np.mean(fold_metrics_10['r2']),
        'r2_std': np.std(fold_metrics_10['r2']),
        'rmse_mean': np.mean(fold_metrics_10['rmse']),
        'rmse_std': np.std(fold_metrics_10['rmse']),
        'mae_mean': np.mean(fold_metrics_10['mae']),
        'mae_std': np.std(fold_metrics_10['mae']),
        'roc_auc_mean': np.mean(fold_metrics_10['roc_auc']),
        'roc_auc_std': np.std(fold_metrics_10['roc_auc']),
        'pr_auc_mean': np.mean(fold_metrics_10['pr_auc']),
        'pr_auc_std': np.std(fold_metrics_10['pr_auc'])
    }

    print(f"\n10-Fold CV Average:")
    print(f"  RÂ²:       {results['10-fold']['r2_mean']:.4f} Â± {results['10-fold']['r2_std']:.4f}")
    print(f"  RMSE:     {results['10-fold']['rmse_mean']:.4f} Â± {results['10-fold']['rmse_std']:.4f}")
    print(f"  ROC-AUC:  {results['10-fold']['roc_auc_mean']:.4f} Â± {results['10-fold']['roc_auc_std']:.4f}")
    print(f"  PR-AUC:   {results['10-fold']['pr_auc_mean']:.4f} Â± {results['10-fold']['pr_auc_std']:.4f}")

    # ========================================
    # Test Set Evaluation
    # ========================================
    print("\nTEST SET EVALUATION")
    print("-" * 70)

    # Train final models on train+val
    test_preds = []
    for name, model in models.items():
        model.fit(X_train_val, y_train_val)
        test_preds.append(model.predict(X_test))

    y_pred_test = np.mean(test_preds, axis=0)

    # Test metrics
    test_r2 = r2_score(y_test, y_pred_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    test_mae = mean_absolute_error(y_test, y_pred_test)

    test_auc_metrics = calculate_auc_metrics(y_test, y_pred_test, mode='binary')

    results['test'] = {
        'r2': test_r2,
        'rmse': test_rmse,
        'mae': test_mae,
        'roc_auc': test_auc_metrics.get('roc_auc_binary', 0.0),
        'pr_auc': test_auc_metrics.get('pr_auc_binary', 0.0)
    }

    print(f"Test Set Performance:")
    print(f"  RÂ²:       {test_r2:.4f}")
    print(f"  RMSE:     {test_rmse:.4f}")
    print(f"  MAE:      {test_mae:.4f}")
    print(f"  ROC-AUC:  {test_auc_metrics.get('roc_auc_binary', 0.0):.4f}")
    print(f"  PR-AUC:   {test_auc_metrics.get('pr_auc_binary', 0.0):.4f}")

    # Plot ROC and PR curves for test set
    roc_auc, pr_auc = plot_auc_curves(
        y_test, y_pred_test,
        f"{plip_dir}/plip_only_roc_pr_curves.png",
        title_prefix='PLIP Only - '
    )

    # Save results
    results_df = pd.DataFrame({
        'Metric': ['RÂ²', 'RMSE', 'MAE', 'ROC-AUC', 'PR-AUC'],
        '5-Fold_Mean': [
            results['5-fold']['r2_mean'],
            results['5-fold']['rmse_mean'],
            results['5-fold']['mae_mean'],
            results['5-fold']['roc_auc_mean'],
            results['5-fold']['pr_auc_mean']
        ],
        '5-Fold_Std': [
            results['5-fold']['r2_std'],
            results['5-fold']['rmse_std'],
            results['5-fold']['mae_std'],
            results['5-fold']['roc_auc_std'],
            results['5-fold']['pr_auc_std']
        ],
        '10-Fold_Mean': [
            results['10-fold']['r2_mean'],
            results['10-fold']['rmse_mean'],
            results['10-fold']['mae_mean'],
            results['10-fold']['roc_auc_mean'],
            results['10-fold']['pr_auc_mean']
        ],
        '10-Fold_Std': [
            results['10-fold']['r2_std'],
            results['10-fold']['rmse_std'],
            results['10-fold']['mae_std'],
            results['10-fold']['roc_auc_std'],
            results['10-fold']['pr_auc_std']
        ],
        'Test': [
            results['test']['r2'],
            results['test']['rmse'],
            results['test']['mae'],
            results['test']['roc_auc'],
            results['test']['pr_auc']
        ]
    })

    results_df.to_csv(f"{plip_dir}/plip_only_performance.csv", index=False)

    print(f"\nâœ“ PLIP-only results saved to {plip_dir}/")
    print("="*70 + "\n")

    return results

    #==============================================================================
    # Optimal Number of Features via CV
    #==============================================================================

def find_optimal_feature_number(X, y, feature_names, prolif_indices,
                                min_features=100, max_features=600,
                                step=50, cv_folds=5, output_dir='gsk3_results'):
    """
    Find optimal number of features through cross-validation

    Args:
        X: Feature matrix after constant removal
        y: Target values
        feature_names: List of feature names
        prolif_indices: Indices of ProLIF features (always protected)
        min_features: Minimum number of features to test (default: 100)
        max_features: Maximum number of features to test (default: 600)
        step: Step size for testing (default: 50)
        cv_folds: Number of CV folds (default: 5)
        output_dir: Directory to save results

    Returns:
        optimal_n: Optimal number of features
        results_df: DataFrame with all test results
    """

    print("\n" + "="*70)
    print("FINDING OPTIMAL NUMBER OF FEATURES")
    print("="*70)
    print(f"Search range: {min_features} to {max_features} (step: {step})")
    print(f"Cross-validation: {cv_folds} folds")
    print(f"ProLIF protected features: {len(prolif_indices)}")
    print("="*70 + "\n")

    # Create output directory
    opt_dir = f"{output_dir}/feature_optimization"
    os.makedirs(opt_dir, exist_ok=True)

    # Test different numbers of features
    n_features_list = list(range(min_features, max_features + 1, step))
    results = []

    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=42)

    for n_features in n_features_list:
        print(f"\n{'='*70}")
        print(f"Testing with {n_features} features...")
        print(f"{'='*70}")

        start_time = time.time()

        # Perform mRMR selection
        print(f"  Selecting features via mRMR...")
        selected_indices, selected_names = mrmr_feature_selection_with_prolif(
            X, y, feature_names,
            n_selected=n_features,
            prolif_indices=prolif_indices
        )

        X_selected = X[:, selected_indices]

        # Cross-validation
        fold_scores = []
        fold_rmses = []

        for fold, (train_idx, val_idx) in enumerate(kfold.split(X_selected), 1):
            X_train, X_val = X_selected[train_idx], X_selected[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            # Train model
            model = GradientBoostingRegressor(
                n_estimators=100, max_depth=6,
                learning_rate=0.1, random_state=42
            )
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            r2 = r2_score(y_val, y_pred)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))

            fold_scores.append(r2)
            fold_rmses.append(rmse)

            print(f"    Fold {fold}: RÂ² = {r2:.4f}, RMSE = {rmse:.4f}")

        mean_r2 = np.mean(fold_scores)
        std_r2 = np.std(fold_scores)
        mean_rmse = np.mean(fold_rmses)
        std_rmse = np.std(fold_rmses)

        elapsed = time.time() - start_time

        results.append({
            'n_features': n_features,
            'r2_mean': mean_r2,
            'r2_std': std_r2,
            'rmse_mean': mean_rmse,
            'rmse_std': std_rmse,
            'time_seconds': elapsed
        })

        print(f"  Average: RÂ² = {mean_r2:.4f} Â± {std_r2:.4f}, "
              f"RMSE = {mean_rmse:.4f} Â± {std_rmse:.4f}")
        print(f"  Time: {elapsed:.1f}s")

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Find optimal number (highest RÂ² with consideration of std)
    results_df['r2_robust'] = results_df['r2_mean'] - results_df['r2_std']
    optimal_idx = results_df['r2_robust'].idxmax()
    optimal_n = results_df.loc[optimal_idx, 'n_features']

    print("\n" + "="*70)
    print("OPTIMIZATION RESULTS")
    print("="*70)
    print(results_df.to_string(index=False))
    print("\n" + "="*70)
    print(f"OPTIMAL NUMBER OF FEATURES: {int(optimal_n)}")
    print(f"Expected RÂ²: {results_df.loc[optimal_idx, 'r2_mean']:.4f} Â± "
          f"{results_df.loc[optimal_idx, 'r2_std']:.4f}")
    print(f"Expected RMSE: {results_df.loc[optimal_idx, 'rmse_mean']:.4f} Â± "
          f"{results_df.loc[optimal_idx, 'rmse_std']:.4f}")
    print("="*70)

    # Save results
    results_df.to_csv(f"{opt_dir}/optimization_results.csv", index=False)

    # Visualization
    print("\nCreating optimization visualizations...")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Plot 1: RÂ² with error bars
    ax1 = axes[0, 0]
    ax1.errorbar(results_df['n_features'], results_df['r2_mean'],
                yerr=results_df['r2_std'], marker='o', capsize=5,
                linewidth=2, markersize=8, color='steelblue')
    ax1.axvline(optimal_n, color='red', linestyle='--', linewidth=2,
               label=f'Optimal: {int(optimal_n)} features')
    ax1.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    ax1.set_ylabel('RÂ² Score', fontsize=12, fontweight='bold')
    ax1.set_title('RÂ² Score vs Number of Features', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Plot 2: RMSE with error bars
    ax2 = axes[0, 1]
    ax2.errorbar(results_df['n_features'], results_df['rmse_mean'],
                yerr=results_df['rmse_std'], marker='s', capsize=5,
                linewidth=2, markersize=8, color='coral')
    ax2.axvline(optimal_n, color='red', linestyle='--', linewidth=2,
               label=f'Optimal: {int(optimal_n)} features')
    ax2.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    ax2.set_ylabel('RMSE', fontsize=12, fontweight='bold')
    ax2.set_title('RMSE vs Number of Features', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    # Plot 3: Robust RÂ² (RÂ² - std)
    ax3 = axes[1, 0]
    ax3.plot(results_df['n_features'], results_df['r2_robust'],
            marker='D', linewidth=2, markersize=8, color='green')
    ax3.axvline(optimal_n, color='red', linestyle='--', linewidth=2,
               label=f'Optimal: {int(optimal_n)} features')
    ax3.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Robust RÂ² (RÂ² - std)', fontsize=12, fontweight='bold')
    ax3.set_title('Robust Performance vs Number of Features',
                 fontsize=13, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)

    # Plot 4: Computation Time
    ax4 = axes[1, 1]
    ax4.bar(results_df['n_features'], results_df['time_seconds'],
           color='purple', alpha=0.6, edgecolor='black')
    ax4.axvline(optimal_n, color='red', linestyle='--', linewidth=2,
               label=f'Optimal: {int(optimal_n)} features')
    ax4.set_xlabel('Number of Features', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Computation Time (seconds)', fontsize=12, fontweight='bold')
    ax4.set_title('Computational Cost vs Number of Features',
                 fontsize=13, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig(f"{opt_dir}/optimization_results.png", dpi=300, bbox_inches='tight')
    plt.close()

    print(f"âœ“ Optimization results saved to {opt_dir}/")

    return int(optimal_n), results_df

In [ ]:
#==============================================================================
# 13. ENHANCED Main Pipeline with mRMR + Ablation Study
#==============================================================================

def run_gsk3_pipeline(input_csv):
    """
    GSK3-Beta QSAR Pipeline
    Enhanced with mRMR Feature Selection + Ablation Study
    """

    print("\n" + "="*70)
    print("GSK3-Beta QSAR MODELING PIPELINE")
    print("="*70)
    print("ENHANCED VERSION with:")
    print("  1. ProLIF-Validated GSK3 Descriptors (8 features, p<0.05)")
    print("  2. mRMR Feature Selection (with ProLIF protection)")
    print("  3. Ablation Study (feature group importance)")
    print("  4. Bayesian-Optimized Ensemble Learning")
    print("  5. SHAP Analysis for Interpretability")
    print("="*70 + "\n")

    # Create output directory
    output_dir = 'gsk3_results'
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(f"{output_dir}/ablation", exist_ok=True)

    pipeline_stats = {
        'data_loading': {},
        'preprocessing': {},
        'features': {},
        'ablation_study': {},
        'feature_selection': {},
        'training': {},
        'performance': {},
        'models': {}
    }

    # Load data
    print("Loading data...")
    try:
        df = pd.read_csv(input_csv, encoding='latin-1')
    except:
        df = pd.read_csv(input_csv)

    print(f"Loaded {len(df)} compounds")
    pipeline_stats['data_loading']['raw_compounds'] = len(df)

    # Cleaning data
    print("\nCleaning data...")
    df = df.dropna(subset=['pChEMBL Value'])
    df = df.dropna(subset=['Smiles'])
    df = df[np.isfinite(df['pChEMBL Value'])]
    df = df.reset_index(drop=True)

    print(f"After cleaning: {len(df)} compounds")
    pipeline_stats['preprocessing']['cleaned_compounds'] = len(df)
    pipeline_stats['preprocessing']['removed_compounds'] = (
        pipeline_stats['data_loading']['raw_compounds'] - len(df)
    )

    # Create molecules
    print("\nCreating molecular structures...")
    molecules_2d = [Chem.MolFromSmiles(smi) for smi in df['Smiles']]

    valid_indices = [i for i, mol in enumerate(molecules_2d) if mol is not None]
    df = df.iloc[valid_indices].reset_index(drop=True)
    molecules_2d = [molecules_2d[i] for i in valid_indices]

    print(f"Valid molecules: {len(molecules_2d)}")
    pipeline_stats['preprocessing']['valid_molecules'] = len(molecules_2d)

    print("Creating 3D structures...")
    molecules_3d = [generate_3d_conformation(mol) for mol in molecules_2d]
    success_3d = sum(1 for mol in molecules_3d if mol is not None and mol.GetNumConformers() > 0)

    pipeline_stats['preprocessing']['3d_success_count'] = success_3d
    pipeline_stats['preprocessing']['3d_success_rate'] = (
        success_3d / len(molecules_2d) * 100 if len(molecules_2d) > 0 else 0
    )

    print(f"3D structure success rate: {success_3d}/{len(molecules_2d)} "
          f"({pipeline_stats['preprocessing']['3d_success_rate']:.2f}%)")

    # Calculate descriptors
    print("\nCalculating molecular features...")

    print("  Computing 2D descriptors...")
    descriptors_2d = [compute_essential_2d_descriptors(mol) for mol in molecules_2d]
    descriptors_2d = np.array(descriptors_2d)
    pipeline_stats['features']['2d_descriptors'] = descriptors_2d.shape[1]

    print("  Computing 3D descriptors...")
    descriptors_3d = [calculate_3d_descriptors(mol) for mol in molecules_3d]
    descriptors_3d = np.array(descriptors_3d)
    pipeline_stats['features']['3d_descriptors'] = descriptors_3d.shape[1]

    print("  Computing GSK3-Î² specific descriptors (8 optimized features)...")
    gsk3_descriptors = [enhanced_gsk3_descriptors(mol) for mol in molecules_2d]
    gsk3_descriptors = np.array(gsk3_descriptors)
    pipeline_stats['features']['gsk3_descriptors'] = gsk3_descriptors.shape[1]

    print("  Computing molecular fingerprints...")
    molecular_fingerprints = [generate_molecular_fingerprint(mol, radius=3, nBits=1024)
                            for mol in molecules_2d]
    molecular_fingerprints = np.array(molecular_fingerprints)
    pipeline_stats['features']['fingerprints'] = molecular_fingerprints.shape[1]

    # Combine features
    print("\nCombining all molecular features...")
    combined_features = np.hstack([
        descriptors_2d,
        descriptors_3d,
        gsk3_descriptors,
        molecular_fingerprints
    ])
    pipeline_stats['features']['total_features'] = combined_features.shape[1]

    # Create feature names
    feature_names = []
    feature_names.extend([f"2D_{i:03d}" for i in range(descriptors_2d.shape[1])])
    feature_names.extend([f"3D_{i:03d}" for i in range(descriptors_3d.shape[1])])
    feature_names.extend(['GSK002_HBondDonorsRatio', 'GSK004_TotalHBondPotential',
                         'GSK007_AromaticCarbocycles', 'GSK008_AromaticHeterocycles',
                         'GSK015_RotatableBonds', 'GSK026_NegativeCharges',
                         'GSK037_BasicNRatio', 'GSK043_AromaticRingFraction'])
    feature_names.extend([f"FP_{i:04d}" for i in range(molecular_fingerprints.shape[1])])

    # Define feature groups for ablation study
    feature_groups = {
        '2D': list(range(0, descriptors_2d.shape[1])),
        '3D': list(range(descriptors_2d.shape[1],
                        descriptors_2d.shape[1] + descriptors_3d.shape[1])),
        'GSK3_ProLIF': list(range(descriptors_2d.shape[1] + descriptors_3d.shape[1],
                                  descriptors_2d.shape[1] + descriptors_3d.shape[1] + 8)),
        'Fingerprints': list(range(descriptors_2d.shape[1] + descriptors_3d.shape[1] + 8,
                                  combined_features.shape[1]))
    }

    # ProLIF indices (GSK3 features)
    prolif_indices = feature_groups['GSK3_ProLIF']

    # Check NaN/Inf
    print("\nChecking for NaN/Inf values in features...")
    nan_mask = np.isnan(combined_features).any(axis=1)
    inf_mask = np.isinf(combined_features).any(axis=1)
    invalid_mask = nan_mask | inf_mask

    if invalid_mask.any():
        n_invalid = invalid_mask.sum()
        print(f"Found {n_invalid} samples with NaN/Inf values. Removing them...")
        valid_mask = ~invalid_mask
        combined_features = combined_features[valid_mask]
        df = df[valid_mask].reset_index(drop=True)
        print(f"Remaining samples: {len(df)}")
        pipeline_stats['preprocessing']['removed_invalid'] = n_invalid
    else:
        pipeline_stats['preprocessing']['removed_invalid'] = 0

    pipeline_stats['preprocessing']['final_compounds'] = len(df)

    print(f"\nTotal molecular features: {combined_features.shape[1]}")
    print(f"  2D descriptors: {descriptors_2d.shape[1]}")
    print(f"  3D descriptors: {descriptors_3d.shape[1]}")
    print(f"  GSK3 descriptors: {gsk3_descriptors.shape[1]} (ProLIF-validated)")
    print(f"  Fingerprints: {molecular_fingerprints.shape[1]}")

    # Target values
    target_values = df['pChEMBL Value'].values
    smiles_list = df['Smiles'].tolist()

    # Final validation
    print("\nFinal data validation...")
    print(f"NaN in target: {np.isnan(target_values).sum()}")
    print(f"NaN in features: {np.isnan(combined_features).sum()}")
    print(f"Inf in features: {np.isinf(combined_features).sum()}")

    # Feature preprocessing
    feature_pipeline = FeaturePreprocessingPipeline()
    processed_features = feature_pipeline.fit_transform(combined_features)

    pipeline_stats['features']['removed_constant'] = np.sum(~feature_pipeline.constant_feature_mask)
    pipeline_stats['features']['processed_features'] = processed_features.shape[1]

    print(f"\nProcessed features shape: {processed_features.shape}")

    if np.isnan(processed_features).any():
        print("WARNING: NaN found in processed features. Replacing with 0...")
        processed_features = np.nan_to_num(processed_features, nan=0.0, posinf=0.0, neginf=0.0)

    # Update feature names and groups after constant removal
    feature_names_filtered = [name for i, name in enumerate(feature_names)
                             if feature_pipeline.constant_feature_mask[i]]

    feature_groups_filtered = {}
    for group_name, indices in feature_groups.items():
        new_indices = []
        for idx in indices:
            if feature_pipeline.constant_feature_mask[idx]:
                new_idx = np.sum(feature_pipeline.constant_feature_mask[:idx+1]) - 1
                new_indices.append(new_idx)
        feature_groups_filtered[group_name] = new_indices

    prolif_indices_filtered = feature_groups_filtered['GSK3_ProLIF']

    print(f"Feature names updated: {len(feature_names_filtered)} features")

    #==========================================================================
    # ABLATION STUDY
    #==========================================================================

    print("\n" + "="*70)
    print("STEP 1: ABLATION STUDY")
    print("="*70)

    ablation_results = perform_ablation_study(
        processed_features,
        target_values,
        feature_groups_filtered,
        feature_names_filtered,
        cv_folds=5,
        output_dir=output_dir
    )

    pipeline_stats['ablation_study']['results'] = ablation_results.to_dict('records')
    pipeline_stats['ablation_study']['baseline_r2'] = ablation_results[
        ablation_results['experiment'] == 'All Features']['r2_mean'].values[0]

    #==========================================================================
    # FIND OPTIMAL NUMBER OF FEATURES
    #==========================================================================

    print("\n" + "="*70)
    print("STEP 2: FINDING OPTIMAL NUMBER OF FEATURES")
    print("="*70)

    optimal_n_features, optimization_results = find_optimal_feature_number(
        X=processed_features,
        y=target_values,
        feature_names=feature_names_filtered,
        prolif_indices=prolif_indices_filtered,
        min_features=100,
        max_features=600,
        step=50,
        cv_folds=5,          # 5-fold CV
        output_dir=output_dir
    )

    pipeline_stats['feature_selection']['optimization_results'] = optimization_results.to_dict('records')
    pipeline_stats['feature_selection']['optimal_n_features'] = optimal_n_features

    #==========================================================================
    # mRMR FEATURE SELECTION
    #==========================================================================

    print("\n" + "="*70)
    print("STEP 3: mRMR FEATURE SELECTION WITH OPTIMAL NUMBER")
    print("="*70)

    selected_indices, selected_names = mrmr_feature_selection_with_prolif(
        processed_features,
        target_values,
        feature_names_filtered,
        n_selected=optimal_n_features,
        prolif_indices=prolif_indices_filtered
    )

    X_selected = processed_features[:, selected_indices]

    pipeline_stats['feature_selection']['method'] = 'mRMR with ProLIF protection'
    pipeline_stats['feature_selection']['n_selected'] = len(selected_indices)
    pipeline_stats['feature_selection']['prolif_protected'] = len(prolif_indices_filtered)
    pipeline_stats['feature_selection']['selected_features'] = selected_names

    # Count selected features by group
    selected_groups_count = {}
    for group_name, group_indices in feature_groups_filtered.items():
        count = sum(1 for idx in selected_indices if idx in group_indices)
        selected_groups_count[group_name] = count
        print(f"  {group_name}: {count} features selected")

    pipeline_stats['feature_selection']['selected_by_group'] = selected_groups_count

    # Save selected feature info
    selected_features_df = pd.DataFrame({
        'feature_index': selected_indices,
        'feature_name': selected_names
    })
    selected_features_df.to_csv(f"{output_dir}/selected_features.csv", index=False)

    # Split data
    print("\n" + "="*70)
    print("STEP 4: DATA SPLITTING")
    print("="*70)

    X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
        X_selected, target_values, range(len(target_values)),
        test_size=0.2, random_state=42
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )

    pipeline_stats['training']['train_samples'] = len(X_train)
    pipeline_stats['training']['validation_samples'] = len(X_val)
    pipeline_stats['training']['test_samples'] = len(X_test)
    pipeline_stats['training']['train_ratio'] = len(X_train) / len(X_selected) * 100
    pipeline_stats['training']['val_ratio'] = len(X_val) / len(X_selected) * 100
    pipeline_stats['training']['test_ratio'] = len(X_test) / len(X_selected) * 100

    print(f"Train: {len(X_train)}, Validation: {len(X_val)}, Test: {len(X_test)}")

    # Train ensemble
    print("\n" + "="*70)
    print("STEP 5: ENSEMBLE MODEL TRAINING")
    print("="*70)

    ensemble_system = EnhancedEnsembleSystem(feature_pipeline)
    ensemble_system.fit(X_train, y_train, X_val, y_val)

    pipeline_stats['models']['ensemble_weights'] = ensemble_system.weights
    pipeline_stats['models']['num_models'] = len(ensemble_system.models)

    # Evaluate on test set
    print("\n" + "="*70)
    print("STEP 6: MODEL EVALUATION")
    print("="*70)

    y_pred = ensemble_system.predict(X_test)

    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)

    pipeline_stats['performance']['test_r2'] = test_r2
    pipeline_stats['performance']['test_rmse'] = test_rmse
    pipeline_stats['performance']['test_mae'] = test_mae

    print(f"\nTest Set Performance:")
    print(f"  RÂ² Score: {test_r2:.4f}")
    print(f"  RMSE: {test_rmse:.4f}")
    print(f"  MAE: {test_mae:.4f}")

    # Save results
    print("\nSaving results...")
    joblib.dump(ensemble_system, f"{output_dir}/ensemble_model.pkl")
    feature_pipeline.save(f"{output_dir}/feature_pipeline.pkl")

    import json
    with open(f"{output_dir}/pipeline_statistics.json", 'w') as f:
        def convert_to_native(obj):
            if isinstance(obj, (np.integer, np.int64, np.int32)):
                return int(obj)
            elif isinstance(obj, (np.floating, np.float64, np.float32)):
                return float(obj)
            elif isinstance(obj, np.ndarray):
                return obj.tolist()
            elif isinstance(obj, dict):
                return {k: convert_to_native(v) for k, v in obj.items()}
            elif isinstance(obj, list):
                return [convert_to_native(v) for v in obj]
            else:
                return obj

        json.dump(convert_to_native(pipeline_stats), f, indent=2)

    print(f"âœ“ Pipeline statistics saved to {output_dir}/pipeline_statistics.json")

    # Save predictions
    test_smiles = [smiles_list[i] for i in test_idx]
    results_df = pd.DataFrame({
        'SMILES': test_smiles,
        'Actual_pChEMBL': y_test,
        'Predicted_pChEMBL': y_pred,
        'Actual_Activity': classify_activity(y_test)[0],
        'Predicted_Activity': classify_activity(y_pred)[0],
        'Prediction_Error': np.abs(y_test - y_pred)
    })

    results_df.to_csv(f"{output_dir}/test_predictions.csv", index=False)

    # ========================================
    # Enhanced Evaluation with AUC Metrics
    # ========================================
    print("\n" + "="*70)
    print("COMPREHENSIVE MODEL EVALUATION WITH AUC METRICS")
    print("="*70)

    # 5-Fold Cross-Validation
    cv_5fold_results = enhanced_cross_validation_with_auc(
        X_train, y_train, ensemble_system, n_folds=5
    )

    # 10-Fold Cross-Validation
    cv_10fold_results = enhanced_cross_validation_with_auc(
        X_train, y_train, ensemble_system, n_folds=10
    )

    # Test Set AUC Metrics
    print(f"\n{'='*70}")
    print("TEST SET EVALUATION WITH AUC METRICS")
    print(f"{'='*70}")

    test_auc_metrics = calculate_auc_metrics(y_test, y_pred, mode='binary')

    print(f"Test RÂ²:       {test_r2:.4f}")
    print(f"Test RMSE:     {test_rmse:.4f}")
    print(f"Test MAE:      {test_mae:.4f}")
    print(f"Test ROC-AUC:  {test_auc_metrics.get('roc_auc_binary', 0.0):.4f}")
    print(f"Test PR-AUC:   {test_auc_metrics.get('pr_auc_binary', 0.0):.4f}")
    print(f"{'='*70}\n")

    # Plot ROC and PR curves
    roc_auc, pr_auc = plot_auc_curves(
        y_test, y_pred,
        f"{output_dir}/roc_pr_curves.png",
        title_prefix='Full Model - '
    )

    # Save comprehensive results
    auc_results_df = pd.DataFrame({
        'Metric': ['RÂ²', 'RMSE', 'MAE', 'ROC-AUC', 'PR-AUC'],
        '5-Fold_Mean': [
            cv_5fold_results['r2_mean'],
            cv_5fold_results['rmse_mean'],
            cv_5fold_results['mae_mean'],
            cv_5fold_results['roc_auc_mean'],
            cv_5fold_results['pr_auc_mean']
        ],
        '5-Fold_Std': [
            cv_5fold_results['r2_std'],
            cv_5fold_results['rmse_std'],
            cv_5fold_results['mae_std'],
            cv_5fold_results['roc_auc_std'],
            cv_5fold_results['pr_auc_std']
        ],
        '10-Fold_Mean': [
            cv_10fold_results['r2_mean'],
            cv_10fold_results['rmse_mean'],
            cv_10fold_results['mae_mean'],
            cv_10fold_results['roc_auc_mean'],
            cv_10fold_results['pr_auc_mean']
        ],
        '10-Fold_Std': [
            cv_10fold_results['r2_std'],
            cv_10fold_results['rmse_std'],
            cv_10fold_results['mae_std'],
            cv_10fold_results['roc_auc_std'],
            cv_10fold_results['pr_auc_std']
        ],
        'Test': [
            test_r2,
            test_rmse,
            test_mae,
            test_auc_metrics.get('roc_auc_binary', 0.0),
            test_auc_metrics.get('pr_auc_binary', 0.0)
        ]
    })

    auc_results_df.to_csv(f"{output_dir}/comprehensive_auc_results.csv", index=False)

    # Update pipeline statistics
    pipeline_stats['performance']['cv_5fold'] = cv_5fold_results
    pipeline_stats['performance']['cv_10fold'] = cv_10fold_results
    pipeline_stats['performance']['test_auc_metrics'] = test_auc_metrics

    # ========================================
    # PLIP-Only Performance Evaluation
    # ========================================
    print("\n" + "="*70)
    print("EVALUATING PLIP (ProLIF) FEATURES ONLY")
    print("="*70)

    # Extract PLIP features
    plip_start = descriptors_2d.shape[1] + descriptors_3d.shape[1]
    plip_end = plip_start + 8
    X_plip_only = combined_features[:, plip_start:plip_end]

    plip_only_results = evaluate_plip_only_performance(
        X_plip_only, target_values, cv_folds=5, output_dir=output_dir
    )

    pipeline_stats['plip_only_performance'] = plip_only_results

    # Visualizations
    print("\nCreating visualizations...")

    # Prediction scatter plot
    plt.figure(figsize=(10, 8))
    plt.scatter(y_test, y_pred, alpha=0.5, s=50, edgecolors='k', linewidth=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', lw=2, label='Perfect prediction')
    plt.xlabel('Actual pChEMBL', fontsize=12, fontweight='bold')
    plt.ylabel('Predicted pChEMBL', fontsize=12, fontweight='bold')
    plt.title(f'GSK3Î² Model - Activity Prediction\n'
              f'RÂ² = {test_r2:.4f}, RMSE = {test_rmse:.4f}',
              fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/prediction_scatter.png", dpi=300, bbox_inches='tight')
    plt.close()

    # Feature importance (if Random Forest available)
    if 'random_forest' in ensemble_system.models:
        rf_model = ensemble_system.models['random_forest']
        importances = rf_model.feature_importances_

        # Adjust feature names if needed
        if len(selected_names) != len(importances):
            print(f"WARNING: Adjusting feature names length...")
            if len(selected_names) > len(importances):
                selected_names_adj = selected_names[:len(importances)]
            else:
                selected_names_adj = selected_names + [f"Feature_{i:04d}"
                                                       for i in range(len(selected_names), len(importances))]
        else:
            selected_names_adj = selected_names

        indices = np.argsort(importances)[::-1][:20]

        plt.figure(figsize=(12, 8))
        plt.barh(range(20), importances[indices], color='steelblue', edgecolor='black')
        plt.yticks(range(20), [selected_names_adj[i] for i in indices])
        plt.xlabel('Feature Importance', fontsize=12, fontweight='bold')
        plt.title('Top 20 Most Important Features', fontsize=14, fontweight='bold')
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"{output_dir}/feature_importance.png", dpi=300, bbox_inches='tight')
        plt.close()

        feature_importance_df = pd.DataFrame({
            'Feature': selected_names_adj,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        feature_importance_df.to_csv(f"{output_dir}/feature_importance.csv", index=False)

        # SHAP analysis
        try:
            perform_shap_analysis(rf_model, X_test, selected_names_adj, output_dir)
        except Exception as e:
            print(f"SHAP analysis failed: {e}")

    # Performance summary
    summary = {
        'model_version': 'GSK3_ProLIF_mRMR_Ablation',
        'total_compounds': len(df),
        'training_samples': len(X_train),
        'validation_samples': len(X_val),
        'test_samples': len(X_test),
        'initial_features': combined_features.shape[1],
        'processed_features': processed_features.shape[1],
        'selected_features': len(selected_indices),
        'gsk3_prolif_features': 8,
        'test_r2': test_r2,
        'test_rmse': test_rmse,
        'test_mae': test_mae,
        'ensemble_weights': ensemble_system.weights,
        'ablation_baseline_r2': ablation_results[
            ablation_results['experiment'] == 'All Features']['r2_mean'].values[0]
    }

    with open(f"{output_dir}/performance_summary.txt", 'w') as f:
        f.write("GSK3-Î² QSAR Model Performance Summary\n")
        f.write("="*70 + "\n\n")
        f.write("ENHANCED with mRMR Feature Selection + Ablation Study\n\n")
        for key, value in summary.items():
            f.write(f"{key}: {value}\n")

    print(f"\nAll results saved to '{output_dir}' directory")
    print("\nPipeline completed successfully!")

    return {
        'ensemble_system': ensemble_system,
        'feature_pipeline': feature_pipeline,
        'performance': {'r2': test_r2, 'rmse': test_rmse, 'mae': test_mae},
        'output_directory': output_dir,
        'pipeline_statistics': pipeline_stats,
        'ablation_results': ablation_results,
        'selected_features': selected_names
    }

In [ ]:
#==============================================================================
# 14. Automated Flowchart Generation
#==============================================================================

def generate_methodology_flowcharts_automated(pipeline_results):
    """
    Generate ALL 3 flowcharts using actual pipeline execution results

    Charts:
    1. Main Pipeline Flowchart (with mRMR + Ablation)
    2. Feature Engineering Detail
    3. Ensemble Architecture Detail

    Args:
        pipeline_results: Dictionary returned by run_gsk3_pipeline()

    Returns:
        flowchart_dir: Path to generated flowcharts
    """
    stats = pipeline_results['pipeline_statistics']
    output_dir = pipeline_results['output_directory']

    try:
        from graphviz import Digraph
    except ImportError:
        print("Installing graphviz...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "graphviz"])
        from graphviz import Digraph

    flowchart_dir = f"{output_dir}/flowcharts"
    os.makedirs(flowchart_dir, exist_ok=True)

    # Extract actual numbers from pipeline execution
    raw_n = stats['data_loading']['raw_compounds']
    cleaned_n = stats['preprocessing']['cleaned_compounds']
    final_n = stats['preprocessing']['final_compounds']

    train_n = stats['training']['train_samples']
    val_n = stats['training']['validation_samples']
    test_n = stats['training']['test_samples']

    train_pct = stats['training']['train_ratio']
    val_pct = stats['training']['val_ratio']
    test_pct = stats['training']['test_ratio']

    total_feat = stats['features']['total_features']
    processed_feat = stats['features']['processed_features']
    removed_const = stats['features']['removed_constant']

    feat_2d = stats['features']['2d_descriptors']
    feat_3d = stats['features']['3d_descriptors']
    feat_gsk3 = stats['features']['gsk3_descriptors']
    feat_fp = stats['features']['fingerprints']
    success_3d_rate = stats['preprocessing']['3d_success_rate']

    test_r2 = pipeline_results['performance']['r2']
    test_rmse = pipeline_results['performance']['rmse']
    test_mae = pipeline_results['performance']['mae']

    weights = stats['models']['ensemble_weights']

    # Get optimal feature info if available
    if 'feature_selection' in stats and 'optimal_n_features' in stats['feature_selection']:
        optimal_n = stats['feature_selection']['optimal_n_features']
        selected_feat = stats['feature_selection']['n_selected']
    else:
        optimal_n = "N/A"
        selected_feat = stats['feature_selection']['n_selected']

    # =========================================================================
    # Main Pipeline Flowchart (with optimization steps)
    # =========================================================================

    print("\nGenerating Figure 1: Main Pipeline Flowchart...")

    main_flow = Digraph('MainPipeline', format='png')
    main_flow.attr(rankdir='TB', size='8,14', dpi='300')
    main_flow.attr('node', shape='box', style='filled', fillcolor='lightblue',
                   fontname='Arial', fontsize='10')

    main_flow.node('A', f'Raw Data\nGSK3Î² Bioassay\n(n={raw_n:,} compounds)',
                   fillcolor='#E8F4F8')

    main_flow.node('B', f'Data Preprocessing\nâ€¢ Remove NaN/Inf values\n'
                        f'â€¢ Filter invalid SMILES\nâ€¢ Validate pChEMBL values\n'
                        f'Removed: {raw_n - cleaned_n} compounds',
                   fillcolor='#FFE6E6')

    main_flow.node('C', f'Cleaned Dataset\n(n={final_n:,} compounds)',
                   fillcolor='#E8F4F8')

    main_flow.node('D', 'Molecular Feature Generation\n(See Figure 2 for details)',
                   fillcolor='#FFF4E6', shape='box', style='filled,rounded')

    main_flow.node('E', f'Feature Preprocessing Pipeline\n'
                        f'â€¢ Remove constant features (variance < 1e-8)\n'
                        f'â€¢ RobustScaler normalization\n'
                        f'{total_feat} â†’ {processed_feat} features\n'
                        f'(Removed {removed_const} constant features)',
                   fillcolor='#FFE6E6')

    main_flow.node('F', f'ABLATION STUDY\n'
                        f'Feature Group Importance Analysis\n'
                        f'5-fold CV for each group\n'
                        f'Quantify contribution of each group',
                   fillcolor='#FFE6F0', style='filled,bold')

    if optimal_n != "N/A":
        main_flow.node('G', f'FEATURE OPTIMIZATION\n'
                            f'Cross-Validation Based Selection\n'
                            f'Tested: 100-600 features\n'
                            f'Optimal: {optimal_n} features',
                       fillcolor='#E6FFE6', style='filled,bold')
    else:
        main_flow.node('G', f'FEATURE OPTIMIZATION\n'
                            f'mRMR Feature Selection\n'
                            f'{processed_feat} â†’ {selected_feat} features',
                       fillcolor='#E6FFE6', style='filled,bold')

    main_flow.node('H', f'Train/Validation/Test Split\n'
                        f'{train_pct:.1f}% / {val_pct:.1f}% / {test_pct:.1f}%\n'
                        f'({train_n:,} / {val_n:,} / {test_n:,})',
                   fillcolor='#E6F3FF')

    main_flow.node('I', 'Ensemble Model Training\n(See Figure 3 for details)',
                   fillcolor='#FFF4E6', shape='box', style='filled,rounded')

    main_flow.node('J', 'Ensemble Weight Optimization\n'
                        'Minimize validation RMSE using SLSQP',
                   fillcolor='#FFE6F0')

    main_flow.node('K', f'Model Evaluation\n'
                        f'Test RÂ² = {test_r2:.4f}\n'
                        f'RMSE = {test_rmse:.4f}\n'
                        f'MAE = {test_mae:.4f}',
                   fillcolor='#E6FFE6')

    main_flow.node('L', 'Model Interpretation\nâ€¢ SHAP analysis\n'
                        'â€¢ Feature importance\nâ€¢ Activity classification',
                   fillcolor='#F0E6FF')

    # Edges
    edges = [('A','B'), ('B','C'), ('C','D'), ('D','E'),
             ('E','F'), ('F','G'), ('G','H'), ('H','I'),
             ('I','J'), ('J','K'), ('K','L')]
    for src, dst in edges:
        main_flow.edge(src, dst)

    main_flow.render(f'{flowchart_dir}/01_main_pipeline_automated', cleanup=True)
    print(f"  âœ“ Saved: 01_main_pipeline_automated.png")

    # =========================================================================
    # Feature Engineering Detail
    # =========================================================================

    print("\nGenerating Figure 2: Feature Engineering Detail...")

    feature_flow = Digraph('FeatureEngineering', format='png')
    feature_flow.attr(rankdir='TB', size='10,14', dpi='300')
    feature_flow.attr('node', shape='box', style='filled', fontname='Arial', fontsize='10')

    feature_flow.node('F0', f'SMILES Strings\n(n={final_n:,})', fillcolor='#E8F4F8')

    feature_flow.node('F1', '2D Molecule Generation\nRDKit: Chem.MolFromSmiles()',
                      fillcolor='#FFE6E6')

    feature_flow.node('F2', f'3D Conformer Generation\n'
                            f'EmbedMolecule + MMFF optimization\n'
                            f'Success rate: {success_3d_rate:.2f}%',
                      fillcolor='#FFE6E6')

    with feature_flow.subgraph(name='cluster_descriptors') as desc:
        desc.attr(label='Parallel Feature Calculation', style='dashed', color='blue')

        desc.node('F3a', f'2D Descriptors ({feat_2d})\nâ€¢ Molecular weight\n'
                         f'â€¢ LogP, TPSA\nâ€¢ H-bond donors/acceptors\n'
                         f'â€¢ Topological indices',
                  fillcolor='#E6F3FF')

        desc.node('F3b', f'3D Descriptors ({feat_3d})\nâ€¢ Asphericity\n'
                         f'â€¢ Radius of gyration\nâ€¢ Shape indices\n'
                         f'â€¢ PMI descriptors',
                  fillcolor='#E6F3FF')

        desc.node('F3c', f'GSK3Î²-Specific ({feat_gsk3})\n'
                         f'ProLIF-validated (p<0.05)\nâ€¢ HBond ratios\n'
                         f'â€¢ Aromatic features\nâ€¢ Charge properties',
                  fillcolor='#FFE6F0', style='filled,bold')

        desc.node('F3d', f'ECFP Fingerprints ({feat_fp})\nMorgan FP, radius=3',
                  fillcolor='#E6F3FF')

    feature_flow.node('F4', f'Feature Concatenation\nTotal: {total_feat:,} features',
                      fillcolor='#E6FFE6')

    # Edges
    edges = [('F0','F1'), ('F1','F2'), ('F2','F3a'), ('F2','F3b'),
             ('F1','F3c'), ('F1','F3d'),
             ('F3a','F4'), ('F3b','F4'), ('F3c','F4'), ('F3d','F4')]
    for src, dst in edges:
        feature_flow.edge(src, dst)

    feature_flow.render(f'{flowchart_dir}/02_feature_engineering_automated', cleanup=True)
    print(f"  âœ“ Saved: 02_feature_engineering_automated.png")

    # =========================================================================
    # Ensemble Architecture
    # =========================================================================

    print("\nGenerating Figure 3: Ensemble Architecture...")

    ensemble_flow = Digraph('EnsembleArchitecture', format='png')
    ensemble_flow.attr(rankdir='TB', size='10,12', dpi='300')
    ensemble_flow.attr('node', shape='box', style='filled', fontname='Arial', fontsize='10')

    ensemble_flow.node('E0', f'Preprocessed Features\n({selected_feat} features)',
                       fillcolor='#E8F4F8')

    # Model hyperparameters for display
    model_params = {
        'random_forest': 'n_estimators=200\nmax_depth=15',
        'xgboost': 'n_estimators=200\nmax_depth=8',
        'svr': 'RBF kernel\nC=10.0',
        'gradient_boosting': 'n_estimators=200\nmax_depth=8',
        'elastic_net': 'alpha=0.1\nl1_ratio=0.5',
        'neural_network': '3 hidden layers\n[256â†’128â†’64]'
    }

    with ensemble_flow.subgraph(name='cluster_models') as models:
        models.attr(label='Base Models', style='dashed', color='darkgreen')

        # Dynamically create nodes based on actual models
        model_nodes = []
        for i, (model_name, weight) in enumerate(weights.items(), 1):
            node_id = f'E{i}'
            model_nodes.append(node_id)

            # Format model name
            display_name = model_name.replace('_', ' ').title()
            weight_pct = weight * 100

            # Get hyperparameters
            params = model_params.get(model_name, '')

            # Highlight top model
            if weight == max(weights.values()):
                fillcolor = '#FFE6F0'
                style = 'filled,bold'
            else:
                fillcolor = '#E6F3FF'
                style = 'filled'

            label = f'{display_name}\n{params}\nWeight: {weight_pct:.1f}%'
            models.node(node_id, label, fillcolor=fillcolor, style=style)

    next_node = len(weights) + 1
    ensemble_flow.node(f'E{next_node}', f'Individual Predictions\n({len(weights)} models)',
                      fillcolor='#FFF4E6')

    ensemble_flow.node(f'E{next_node+1}', 'Weighted Ensemble\n'
                                          'Weights optimized via SLSQP\n'
                                          'Minimize validation loss',
                       fillcolor='#FFE6F0')

    ensemble_flow.node(f'E{next_node+2}', 'Final Prediction\npChEMBL Value',
                       fillcolor='#E6FFE6', shape='ellipse', style='filled,bold')

    # Edges
    for i in range(1, len(weights) + 1):
        ensemble_flow.edge('E0', f'E{i}')
        ensemble_flow.edge(f'E{i}', f'E{next_node}')

    ensemble_flow.edge(f'E{next_node}', f'E{next_node+1}')
    ensemble_flow.edge(f'E{next_node+1}', f'E{next_node+2}')

    ensemble_flow.render(f'{flowchart_dir}/03_ensemble_architecture_automated', cleanup=True)
    print(f"  âœ“ Saved: 03_ensemble_architecture_automated.png")

    # =========================================================================
    # Summary
    # =========================================================================

    print(f"\n{'='*70}")
    print(f"âœ“ All 3 AUTOMATED flowcharts saved to: {flowchart_dir}/")
    print(f"{'='*70}")
    print("\nGenerated files:")
    print("  1. 01_main_pipeline_automated.png - Overall methodology")
    print("  2. 02_feature_engineering_automated.png - Feature generation")
    print("  3. 03_ensemble_architecture_automated.png - Model ensemble")

    return flowchart_dir

In [ ]:
#==============================================================================
# 15. Feature Transformation Flowchart
#==============================================================================

def generate_feature_transformation_flowchart(pipeline_results):
    """
    Generate detailed Feature Transformation Pipeline showing
    the complete evolution of features from raw to final selection

    Args:
        pipeline_results: Dictionary returned by run_gsk3_pipeline()

    Returns:
        flowchart_path: Path to generated flowchart
    """
    try:
        from graphviz import Digraph
    except ImportError:
        print("Installing graphviz...")
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "graphviz"])
        from graphviz import Digraph

    stats = pipeline_results['pipeline_statistics']
    output_dir = pipeline_results['output_directory']

    # Extract numbers
    feat_2d = stats['features']['2d_descriptors']
    feat_3d = stats['features']['3d_descriptors']
    feat_gsk3 = stats['features']['gsk3_descriptors']
    feat_fp = stats['features']['fingerprints']
    total_initial = stats['features']['total_features']
    removed_const = stats['features']['removed_constant']
    processed_feat = stats['features']['processed_features']

    # Feature selection info
    if 'feature_selection' in stats:
        selected_feat = stats['feature_selection']['n_selected']
        prolif_protected = stats['feature_selection']['prolif_protected']
        selected_by_group = stats['feature_selection'].get('selected_by_group', {})
        optimal_n = stats['feature_selection'].get('optimal_n_features', selected_feat)
    else:
        selected_feat = processed_feat
        prolif_protected = feat_gsk3
        selected_by_group = {}
        optimal_n = selected_feat

    # Create flowchart
    flow = Digraph('FeatureTransformation', format='png')
    flow.attr(rankdir='TB', size='12,16', dpi='300')
    flow.attr('node', shape='box', style='filled', fontname='Arial', fontsize='11')

    # Title
    flow.node('TITLE', 'FEATURE TRANSFORMATION PIPELINE\nFrom Raw Features to Optimized Selection',
              shape='box', style='filled,bold', fillcolor='#E8F4F8',
              fontsize='14', fontname='Arial Bold')

    # Step 1: Initial Feature Generation
    flow.node('S1', f'STEP 1: Feature Generation\n'
                    f'Raw molecular features from SMILES',
              fillcolor='#FFE6E6', shape='box', style='filled,bold')

    with flow.subgraph(name='cluster_generation') as gen:
        gen.attr(label='Parallel Calculation', style='dashed', color='blue')
        gen.node('F1', f'2D Descriptors\nn = {feat_2d}', fillcolor='#E6F3FF')
        gen.node('F2', f'3D Descriptors\nn = {feat_3d}', fillcolor='#E6F3FF')
        gen.node('F3', f'GSK3Î² ProLIF\nn = {feat_gsk3}\n(p < 0.05)',
                fillcolor='#FFE6F0', style='filled,bold')
        gen.node('F4', f'ECFP Fingerprints\nn = {feat_fp}', fillcolor='#E6F3FF')

    flow.node('T1', f'TOTAL FEATURES\nn = {total_initial:,}\n'
                    f'({feat_2d} + {feat_3d} + {feat_gsk3} + {feat_fp})',
              fillcolor='#E6FFE6', shape='box', style='filled,bold')

    # Step 2: Preprocessing
    flow.node('S2', 'STEP 2: Feature Preprocessing\n'
                    'Remove constant & normalize',
              fillcolor='#FFE6E6', shape='box', style='filled,bold')

    flow.node('P1', f'Remove Constant Features\n'
                    f'Variance threshold = 1e-8\n'
                    f'Removed: {removed_const} features',
              fillcolor='#FFF4E6')

    flow.node('P2', f'RobustScaler Normalization\n'
                    f'Robust to outliers\n'
                    f'Remaining: {processed_feat:,} features',
              fillcolor='#FFF4E6')

    flow.node('T2', f'PREPROCESSED FEATURES\nn = {processed_feat:,}\n'
                    f'({total_initial:,} â†’ {processed_feat:,})',
              fillcolor='#E6FFE6', shape='box', style='filled,bold')

    # Step 3: Ablation Study
    flow.node('S3', 'STEP 3: Ablation Study\n'
                    'Quantify feature group importance',
              fillcolor='#FFE6E6', shape='box', style='filled,bold')

    flow.node('A1', 'Test All Feature Groups\n'
                    'â€¢ Remove each group systematically\n'
                    'â€¢ Measure RÂ² decrease\n'
                    'â€¢ 5-fold cross-validation',
              fillcolor='#F0E6FF')

    ablation_info = "Feature Group Importance:\n"
    if 'ablation_study' in stats and 'results' in stats['ablation_study']:
        ablation_results = stats['ablation_study']['results']
        # Sort by RÂ² decrease (skip "All Features")
        sorted_results = sorted(
            [r for r in ablation_results if r['experiment'] != 'All Features'],
            key=lambda x: x['r2_decrease'],
            reverse=True
        )[:3]  # Top 3
        for i, result in enumerate(sorted_results, 1):
            exp_name = result['experiment'].replace('No ', '')
            r2_dec = result['r2_decrease']
            ablation_info += f"{i}. {exp_name}: Î”RÂ² = {r2_dec:.4f}\n"

    flow.node('A2', ablation_info.strip(), fillcolor='#F0E6FF')

    # Step 4: Feature Optimization
    flow.node('S4', 'STEP 4: Feature Number Optimization\n'
                    'Find optimal feature count via CV',
              fillcolor='#FFE6E6', shape='box', style='filled,bold')

    flow.node('O1', f'Test Range: 100 to 600 features\n'
                    f'Step size: 50 features\n'
                    f'Method: 5-fold cross-validation\n'
                    f'Criterion: Maximize (RÂ² - std)',
              fillcolor='#E6F0FF')

    flow.node('O2', f'OPTIMAL FEATURE NUMBER\nn = {optimal_n}\n'
                    f'(Validated by CV)',
              fillcolor='#E6FFE6', shape='box', style='filled,bold')

    # Step 5: Feature Selection
    flow.node('S5', 'STEP 5: mRMR Feature Selection\n'
                    'Minimum Redundancy Maximum Relevance',
              fillcolor='#FFE6E6', shape='box', style='filled,bold')

    flow.node('M1', f'mRMR Algorithm\n'
                    f'â€¢ Maximize relevance (MI with target)\n'
                    f'â€¢ Minimize redundancy (inter-correlation)\n'
                    f'â€¢ Protect ProLIF features (n={prolif_protected})\n'
                    f'â€¢ Greedy selection',
              fillcolor='#E6F0FF')

    # Show selection by group if available
    if selected_by_group:
        selection_text = "Selected by Group:\n"
        for group, count in selected_by_group.items():
            selection_text += f"â€¢ {group}: {count}\n"
        flow.node('M2', selection_text.strip(), fillcolor='#E6F0FF')

    flow.node('T3', f'FINAL FEATURES\nn = {selected_feat}\n'
                    f'({processed_feat:,} â†’ {selected_feat})\n'
                    f'Reduction: {(1-selected_feat/processed_feat)*100:.1f}%',
              fillcolor='#E6FFE6', shape='box', style='filled,bold')

    # Summary box
    flow.node('SUMMARY',
              f'FEATURE TRANSFORMATION SUMMARY\n'
              f'â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”\n'
              f'Initial:           {total_initial:,} features\n'
              f'After Preprocessing:  {processed_feat:,} features (-{removed_const})\n'
              f'After Optimization:   {selected_feat} features (-{processed_feat-selected_feat:,})\n'
              f'â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”\n'
              f'Total Reduction:  {(1-selected_feat/total_initial)*100:.1f}%\n'
              f'ProLIF Protected: {prolif_protected} features (always included)',
              shape='box', style='filled,bold', fillcolor='#FFFFCC',
              fontsize='12', fontname='Courier')

    # Edges - Main flow
    flow.edge('TITLE', 'S1')
    flow.edge('S1', 'F1')
    flow.edge('S1', 'F2')
    flow.edge('S1', 'F3')
    flow.edge('S1', 'F4')
    flow.edge('F1', 'T1')
    flow.edge('F2', 'T1')
    flow.edge('F3', 'T1')
    flow.edge('F4', 'T1')

    flow.edge('T1', 'S2')
    flow.edge('S2', 'P1')
    flow.edge('P1', 'P2')
    flow.edge('P2', 'T2')

    flow.edge('T2', 'S3')
    flow.edge('S3', 'A1')
    flow.edge('A1', 'A2')

    flow.edge('A2', 'S4')
    flow.edge('S4', 'O1')
    flow.edge('O1', 'O2')

    flow.edge('O2', 'S5')
    flow.edge('S5', 'M1')
    if selected_by_group:
        flow.edge('M1', 'M2')
        flow.edge('M2', 'T3')
    else:
        flow.edge('M1', 'T3')

    flow.edge('T3', 'SUMMARY')

    # Save
    flowchart_dir = f"{output_dir}/flowcharts"
    os.makedirs(flowchart_dir, exist_ok=True)

    output_path = f'{flowchart_dir}/04_feature_transformation_detailed'
    flow.render(output_path, cleanup=True)

    print(f"\n{'='*70}")
    print(f"âœ“ Feature Transformation Flowchart Generated!")
    print(f"{'='*70}")
    print(f"Saved to: {output_path}.png")

    return f"{output_path}.png"

In [ ]:
#==============================================================================
# 16.Visualizations
#==============================================================================

def generate_activity_distribution_comparison(y_true, y_pred, output_dir):
    """
    Generate activity distribution comparison figure
    Compares actual vs predicted activity class distributions

    Returns:
        figure_path: Path to saved figure
    """
    print("\nGenerating Activity Distribution Comparison...")

    # Classify activities
    actual_activities = classify_activity(y_true)
    predicted_activities = classify_activity(y_pred)

    # Count distributions
    from collections import Counter
    actual_counts = Counter(actual_activities)
    predicted_counts = Counter(predicted_activities)

    # Activity classes
    classes = ['inactive', 'moderately_active', 'highly_active']
    class_labels = ['Inactive\n(pChEMBL < 5.5)',
                    'Moderately Active\n(5.5 â‰¤ pChEMBL < 7.5)',
                    'Highly Active\n(pChEMBL â‰¥ 7.5)']

    actual_values = [actual_counts.get(cls, 0) for cls in classes]
    predicted_values = [predicted_counts.get(cls, 0) for cls in classes]

    # Calculate percentages
    total = len(y_true)
    actual_pct = [v/total*100 for v in actual_values]
    predicted_pct = [v/total*100 for v in predicted_values]

    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Bar chart comparison
    ax1 = axes[0]
    x = np.arange(len(classes))
    width = 0.35

    bars1 = ax1.bar(x - width/2, actual_values, width, label='Actual',
                    color='steelblue', alpha=0.8, edgecolor='black')
    bars2 = ax1.bar(x + width/2, predicted_values, width, label='Predicted',
                    color='coral', alpha=0.8, edgecolor='black')

    ax1.set_xlabel('Activity Class', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Compounds', fontsize=12, fontweight='bold')
    ax1.set_title('Activity Distribution: Actual vs Predicted',
                  fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(class_labels, fontsize=10)
    ax1.legend(fontsize=11)
    ax1.grid(axis='y', alpha=0.3)

    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}',
                    ha='center', va='bottom', fontweight='bold')

    # Plot 2: Percentage comparison
    ax2 = axes[1]
    x_pos = np.arange(len(classes))

    ax2.plot(x_pos, actual_pct, marker='o', linewidth=2, markersize=10,
            label='Actual', color='steelblue')
    ax2.plot(x_pos, predicted_pct, marker='s', linewidth=2, markersize=10,
            label='Predicted', color='coral')

    ax2.set_xlabel('Activity Class', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
    ax2.set_title('Activity Distribution (Percentage)',
                  fontsize=13, fontweight='bold')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(class_labels, fontsize=10)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    # Add value labels
    for i, (a_pct, p_pct) in enumerate(zip(actual_pct, predicted_pct)):
        ax2.text(i, a_pct + 1, f'{a_pct:.1f}%', ha='center', fontweight='bold')
        ax2.text(i, p_pct - 2, f'{p_pct:.1f}%', ha='center', fontweight='bold')

    plt.tight_layout()

    # Save
    figure_path = f"{output_dir}/activity_distribution_comparison.png"
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Save statistics
    stats_df = pd.DataFrame({
        'Activity_Class': class_labels,
        'Actual_Count': actual_values,
        'Actual_Percentage': actual_pct,
        'Predicted_Count': predicted_values,
        'Predicted_Percentage': predicted_pct,
        'Difference': [p-a for a, p in zip(actual_values, predicted_values)]
    })
    stats_df.to_csv(f"{output_dir}/activity_distribution_stats.csv", index=False)

    print(f"âœ“ Activity distribution comparison saved to {figure_path}")

    return figure_path


def generate_confusion_matrix_figure(y_true, y_pred, output_dir):
    """
    Generate confusion matrix for activity classification
    """
    from sklearn.metrics import confusion_matrix, classification_report

    print("\nGenerating Activity Classification Confusion Matrix...")

    # Classify activities
    actual_activities = classify_activity(y_true)
    predicted_activities = classify_activity(y_pred)

    # Classes
    classes = ['inactive', 'moderately_active', 'highly_active']
    class_labels = ['Inactive', 'Moderately\nActive', 'Highly\nActive']

    # Compute confusion matrix
    cm = confusion_matrix(actual_activities, predicted_activities, labels=classes)

    # Normalize
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Absolute counts
    ax1 = axes[0]
    im1 = ax1.imshow(cm, interpolation='nearest', cmap='Blues')
    ax1.set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
    plt.colorbar(im1, ax=ax1)

    ax1.set_xticks(np.arange(len(classes)))
    ax1.set_yticks(np.arange(len(classes)))
    ax1.set_xticklabels(class_labels)
    ax1.set_yticklabels(class_labels)
    ax1.set_xlabel('Predicted Activity', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Actual Activity', fontsize=12, fontweight='bold')

    # Add text annotations
    for i in range(len(classes)):
        for j in range(len(classes)):
            text = ax1.text(j, i, f'{cm[i, j]}',
                           ha="center", va="center",
                           color="white" if cm[i, j] > cm.max()/2 else "black",
                           fontsize=14, fontweight='bold')

    # Plot 2: Normalized (percentages)
    ax2 = axes[1]
    im2 = ax2.imshow(cm_normalized, interpolation='nearest', cmap='Greens')
    ax2.set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
    plt.colorbar(im2, ax=ax2)

    ax2.set_xticks(np.arange(len(classes)))
    ax2.set_yticks(np.arange(len(classes)))
    ax2.set_xticklabels(class_labels)
    ax2.set_yticklabels(class_labels)
    ax2.set_xlabel('Predicted Activity', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Actual Activity', fontsize=12, fontweight='bold')

    # Add text annotations
    for i in range(len(classes)):
        for j in range(len(classes)):
            text = ax2.text(j, i, f'{cm_normalized[i, j]:.2f}',
                           ha="center", va="center",
                           color="white" if cm_normalized[i, j] > 0.5 else "black",
                           fontsize=14, fontweight='bold')

    plt.tight_layout()

    # Save
    figure_path = f"{output_dir}/activity_confusion_matrix.png"
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Generate classification report
    report = classification_report(actual_activities, predicted_activities,
                                   labels=classes, target_names=class_labels,
                                   output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(f"{output_dir}/activity_classification_report.csv")

    print(f"âœ“ Confusion matrix saved to {figure_path}")

    return figure_path


def generate_residual_analysis_figure(y_true, y_pred, output_dir):
    """
    Generate comprehensive residual analysis figure
    """
    print("\nGenerating Residual Analysis...")

    residuals = y_pred - y_true

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Plot 1: Residual vs Predicted
    ax1 = axes[0, 0]
    ax1.scatter(y_pred, residuals, alpha=0.5, s=30, edgecolors='k', linewidth=0.5)
    ax1.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax1.set_xlabel('Predicted pChEMBL', fontsize=11, fontweight='bold')
    ax1.set_ylabel('Residuals (Predicted - Actual)', fontsize=11, fontweight='bold')
    ax1.set_title('Residual Plot', fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # Plot 2: Residual Distribution
    ax2 = axes[0, 1]
    ax2.hist(residuals, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=0, color='r', linestyle='--', linewidth=2)
    ax2.set_xlabel('Residuals', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Frequency', fontsize=11, fontweight='bold')
    ax2.set_title('Residual Distribution', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')

    # Add statistics
    mean_res = np.mean(residuals)
    std_res = np.std(residuals)
    ax2.text(0.05, 0.95, f'Mean: {mean_res:.4f}\nStd: {std_res:.4f}',
            transform=ax2.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
            fontsize=10, fontweight='bold')

    # Plot 3: Q-Q Plot
    ax3 = axes[1, 0]
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=ax3)
    ax3.set_title('Q-Q Plot (Normality Check)', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)

    # Plot 4: Absolute Residuals vs Predicted
    ax4 = axes[1, 1]
    abs_residuals = np.abs(residuals)
    ax4.scatter(y_pred, abs_residuals, alpha=0.5, s=30,
               edgecolors='k', linewidth=0.5, color='coral')
    ax4.set_xlabel('Predicted pChEMBL', fontsize=11, fontweight='bold')
    ax4.set_ylabel('Absolute Residuals', fontsize=11, fontweight='bold')
    ax4.set_title('Absolute Residual Plot', fontsize=12, fontweight='bold')
    ax4.grid(True, alpha=0.3)

    # Add trend line
    z = np.polyfit(y_pred, abs_residuals, 1)
    p = np.poly1d(z)
    ax4.plot(sorted(y_pred), p(sorted(y_pred)), "r--", linewidth=2,
            label=f'Trend: y={z[0]:.4f}x+{z[1]:.4f}')
    ax4.legend(fontsize=9)

    plt.tight_layout()

    # Save
    figure_path = f"{output_dir}/residual_analysis.png"
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"âœ“ Residual analysis saved to {figure_path}")

    return figure_path


def generate_model_performance_comparison(pipeline_results, output_dir):
    """
    Generate model performance comparison figure
    """
    print("\nGenerating Model Performance Comparison...")

    stats = pipeline_results['pipeline_statistics']
    weights = stats['models']['ensemble_weights']

    # Model names and weights
    model_names = list(weights.keys())
    model_weights = list(weights.values())

    # Create display names
    display_names = [name.replace('_', ' ').title() for name in model_names]

    # Sort by weight
    sorted_indices = np.argsort(model_weights)[::-1]
    model_names_sorted = [display_names[i] for i in sorted_indices]
    weights_sorted = [model_weights[i] for i in sorted_indices]

    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: Model weights
    ax1 = axes[0]
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(model_names_sorted)))
    bars = ax1.barh(model_names_sorted, weights_sorted, color=colors,
                    edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('Ensemble Weight', fontsize=12, fontweight='bold')
    ax1.set_title('Model Contribution to Ensemble', fontsize=13, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    # Add percentage labels
    for i, (bar, weight) in enumerate(zip(bars, weights_sorted)):
        ax1.text(weight + 0.01, bar.get_y() + bar.get_height()/2,
                f'{weight*100:.1f}%', va='center', fontweight='bold', fontsize=10)

    # Plot 2: Performance metrics
    ax2 = axes[1]
    test_r2 = pipeline_results['performance']['r2']
    test_rmse = pipeline_results['performance']['rmse']
    test_mae = pipeline_results['performance']['mae']

    metrics = ['RÂ²', 'RMSE', 'MAE']
    values = [test_r2, test_rmse, test_mae]
    colors_metrics = ['#4CAF50', '#FF9800', '#F44336']

    bars = ax2.bar(metrics, values, color=colors_metrics, alpha=0.7,
                  edgecolor='black', linewidth=1.5)
    ax2.set_ylabel('Value', fontsize=12, fontweight='bold')
    ax2.set_title('Ensemble Performance Metrics', fontsize=13, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)

    # Add value labels
    for bar, value in zip(bars, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.4f}', ha='center', va='bottom',
                fontweight='bold', fontsize=11)

    plt.tight_layout()

    # Save
    figure_path = f"{output_dir}/model_performance_comparison.png"
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"âœ“ Model performance comparison saved to {figure_path}")

    return figure_path


def generate_feature_group_contribution(pipeline_results, output_dir):
    """
    Generate feature group contribution analysis figure
    """
    print("\nGenerating Feature Group Contribution Analysis...")

    stats = pipeline_results['pipeline_statistics']

    # Get ablation study results if available
    if 'ablation_study' not in stats or 'results' not in stats['ablation_study']:
        print("  Ablation study results not available, skipping...")
        return None

    ablation_results = pd.DataFrame(stats['ablation_study']['results'])

    # Filter out "All Features" baseline
    ablation_filtered = ablation_results[ablation_results['experiment'] != 'All Features'].copy()
    ablation_filtered = ablation_filtered.sort_values('r2_decrease', ascending=False)

    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Plot 1: RÂ² decrease (importance)
    ax1 = axes[0]
    colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.9, len(ablation_filtered)))
    bars = ax1.barh(ablation_filtered['experiment'],
                    ablation_filtered['r2_decrease'],
                    color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_xlabel('RÂ² Decrease (Importance)', fontsize=12, fontweight='bold')
    ax1.set_title('Feature Group Importance (Ablation Study)',
                  fontsize=13, fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    # Add value labels
    for bar, value in zip(bars, ablation_filtered['r2_decrease']):
        ax1.text(value + 0.001, bar.get_y() + bar.get_height()/2,
                f'{value:.4f}', va='center', fontweight='bold', fontsize=10)

    # Plot 2: Performance comparison
    ax2 = axes[1]
    baseline_r2 = stats['ablation_study']['baseline_r2']

    # Create data for plotting
    experiments = ['Baseline'] + ablation_filtered['experiment'].tolist()
    r2_values = [baseline_r2] + ablation_filtered['r2_mean'].tolist()

    colors = ['green'] + ['coral'] * len(ablation_filtered)
    bars = ax2.bar(range(len(experiments)), r2_values,
                  color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)

    ax2.set_xticks(range(len(experiments)))
    ax2.set_xticklabels(experiments, rotation=45, ha='right', fontsize=9)
    ax2.set_ylabel('RÂ² Score', fontsize=12, fontweight='bold')
    ax2.set_title('Model Performance with Feature Groups Removed',
                  fontsize=13, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    ax2.axhline(y=baseline_r2, color='green', linestyle='--',
               linewidth=2, alpha=0.5, label='Baseline')
    ax2.legend(fontsize=10)

    # Add value labels
    for bar, value in zip(bars, r2_values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{value:.4f}', ha='center', va='bottom',
                fontweight='bold', fontsize=9)

    plt.tight_layout()

    # Save
    figure_path = f"{output_dir}/feature_group_contribution.png"
    plt.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"âœ“ Feature group contribution saved to {figure_path}")

    return figure_path


def generate_all_publication_figures(pipeline_results, y_test, y_pred):
    """
    Generate all publication-ready figures in one function

    Args:
        pipeline_results: Results from run_gsk3_pipeline()
        y_test: Actual test values
        y_pred: Predicted test values

    Returns:
        figure_paths: Dictionary of all generated figure paths
    """
    output_dir = pipeline_results['output_directory']

    print("\n" + "="*70)
    print("GENERATING ALL PUBLICATION-READY FIGURES")
    print("="*70)

    figure_paths = {}

    try:
        figure_paths['activity_distribution'] = generate_activity_distribution_comparison(
            y_test, y_pred, output_dir
        )
    except Exception as e:
        print(f"âœ— Activity distribution failed: {e}")

    try:
        figure_paths['confusion_matrix'] = generate_confusion_matrix_figure(
            y_test, y_pred, output_dir
        )
    except Exception as e:
        print(f"âœ— Confusion matrix failed: {e}")

    try:
        figure_paths['residual_analysis'] = generate_residual_analysis_figure(
            y_test, y_pred, output_dir
        )
    except Exception as e:
        print(f"âœ— Residual analysis failed: {e}")

    try:
        figure_paths['model_performance'] = generate_model_performance_comparison(
            pipeline_results, output_dir
        )
    except Exception as e:
        print(f"âœ— Model performance comparison failed: {e}")

    try:
        figure_paths['feature_contribution'] = generate_feature_group_contribution(
            pipeline_results, output_dir
        )
    except Exception as e:
        print(f"âœ— Feature group contribution failed: {e}")

    # â˜… NEW: Advanced classification metrics
    try:
        classification_metrics = compute_multiclass_classification_metrics(
            y_test, y_pred, output_dir
        )
        figure_paths['roc_pr_curves'] = f"{output_dir}/roc_pr_curves.png"
        figure_paths['classification_summary'] = f"{output_dir}/classification_metrics_summary.png"

        # Store metrics in pipeline results
        pipeline_results['classification_metrics'] = classification_metrics

    except Exception as e:
        print(f"âœ— Advanced classification metrics failed: {e}")
        import traceback
        traceback.print_exc()

    print("\n" + "="*70)
    print("PUBLICATION FIGURES SUMMARY")
    print("="*70)
    print(f"Successfully generated {len([p for p in figure_paths.values() if p])} figures:")
    for name, path in figure_paths.items():
        if path:
            print(f"  âœ“ {name}")
    print("="*70)

    return figure_paths

In [ ]:
#==============================================================================
# 17. Advanced Classification Metrics and Visualization
#==============================================================================

def compute_multiclass_classification_metrics(y_true, y_pred, output_dir):
    """
    Compute comprehensive classification metrics for activity prediction
    including ROC-AUC, PR-AUC, and detailed performance statistics

    Args:
        y_true: Actual pChEMBL values
        y_pred: Predicted pChEMBL values
        output_dir: Directory to save results

    Returns:
        metrics_dict: Dictionary containing all computed metrics
    """
    from sklearn.metrics import (
        roc_curve, auc, precision_recall_curve, average_precision_score,
        recall_score, precision_score, f1_score, balanced_accuracy_score,
        cohen_kappa_score, matthews_corrcoef
    )
    from sklearn.preprocessing import label_binarize

    print("\n" + "="*70)
    print("COMPUTING ADVANCED CLASSIFICATION METRICS")
    print("="*70)

    # Classify activities
    actual_classes, _ = classify_activity(y_true)
    predicted_classes, _ = classify_activity(y_pred)

    # Define class labels
    class_names = ['inactive', 'moderately_active', 'highly_active']
    class_labels = ['Inactive\n(<5.5)', 'Moderately\nActive\n(5.5-7.5)',
                    'Highly\nActive\n(â‰¥7.5)']
    n_classes = len(class_names)

    # Convert to numeric labels
    label_map = {name: idx for idx, name in enumerate(class_names)}
    y_true_numeric = np.array([label_map[cls] for cls in actual_classes])
    y_pred_numeric = np.array([label_map[cls] for cls in predicted_classes])

    # Binarize labels for ROC and PR curves (one-vs-rest)
    y_true_bin = label_binarize(y_true_numeric, classes=range(n_classes))

    # For predicted probabilities, use proximity to class thresholds
    # This simulates probability scores from regression output
    y_pred_proba = np.zeros((len(y_pred), n_classes))
    thresholds = [5.5, 7.5]

    for i, pred_val in enumerate(y_pred):
        if pred_val < thresholds[0]:
            # Closer to inactive
            distance_to_inactive = abs(pred_val - 4.0)
            y_pred_proba[i, 0] = 1.0 / (1.0 + distance_to_inactive)
            y_pred_proba[i, 1] = (1.0 - y_pred_proba[i, 0]) * 0.7
            y_pred_proba[i, 2] = 1.0 - y_pred_proba[i, 0] - y_pred_proba[i, 1]
        elif pred_val < thresholds[1]:
            # Moderately active region
            distance_to_center = abs(pred_val - 6.5)
            y_pred_proba[i, 1] = 1.0 / (1.0 + distance_to_center)
            y_pred_proba[i, 0] = (1.0 - y_pred_proba[i, 1]) * (thresholds[1] - pred_val) / (thresholds[1] - thresholds[0])
            y_pred_proba[i, 2] = 1.0 - y_pred_proba[i, 0] - y_pred_proba[i, 1]
        else:
            # Highly active
            distance_to_high = abs(pred_val - 8.5)
            y_pred_proba[i, 2] = 1.0 / (1.0 + distance_to_high * 0.5)
            y_pred_proba[i, 1] = (1.0 - y_pred_proba[i, 2]) * 0.6
            y_pred_proba[i, 0] = 1.0 - y_pred_proba[i, 1] - y_pred_proba[i, 2]

    # Normalize probabilities
    y_pred_proba = y_pred_proba / y_pred_proba.sum(axis=1, keepdims=True)

    # ========================================
    # Compute Per-Class Metrics
    # ========================================

    print("\nComputing per-class metrics...")

    per_class_metrics = {}

    for idx, class_name in enumerate(class_names):
        print(f"\n  Class: {class_name}")

        # Binary classification for this class
        y_true_class = (y_true_numeric == idx).astype(int)
        y_pred_class = (y_pred_numeric == idx).astype(int)
        y_score_class = y_pred_proba[:, idx]

        # ROC curve
        fpr, tpr, _ = roc_curve(y_true_class, y_score_class)
        roc_auc = auc(fpr, tpr)

        # Precision-Recall curve
        precision_curve, recall_curve, _ = precision_recall_curve(
            y_true_class, y_score_class
        )
        pr_auc = auc(recall_curve, precision_curve)
        avg_precision = average_precision_score(y_true_class, y_score_class)

        # Store metrics
        per_class_metrics[class_name] = {
            'fpr': fpr,
            'tpr': tpr,
            'roc_auc': roc_auc,
            'precision_curve': precision_curve,
            'recall_curve': recall_curve,
            'pr_auc': pr_auc,
            'avg_precision': avg_precision
        }

        print(f"    ROC-AUC: {roc_auc:.4f}")
        print(f"    PR-AUC: {pr_auc:.4f}")
        print(f"    Avg Precision: {avg_precision:.4f}")

    # ========================================
    # Compute Overall Metrics
    # ========================================

    print("\n" + "-"*70)
    print("Computing overall classification metrics...")

    # Macro-averaged metrics
    precision_macro = precision_score(y_true_numeric, y_pred_numeric, average='macro')
    recall_macro = recall_score(y_true_numeric, y_pred_numeric, average='macro')
    f1_macro = f1_score(y_true_numeric, y_pred_numeric, average='macro')

    # Weighted-averaged metrics
    precision_weighted = precision_score(y_true_numeric, y_pred_numeric, average='weighted')
    recall_weighted = recall_score(y_true_numeric, y_pred_numeric, average='weighted')
    f1_weighted = f1_score(y_true_numeric, y_pred_numeric, average='weighted')

    # Per-class metrics
    precision_per_class = precision_score(y_true_numeric, y_pred_numeric, average=None)
    recall_per_class = recall_score(y_true_numeric, y_pred_numeric, average=None)
    f1_per_class = f1_score(y_true_numeric, y_pred_numeric, average=None)

    # Balanced accuracy
    balanced_acc = balanced_accuracy_score(y_true_numeric, y_pred_numeric)

    # Cohen's Kappa
    kappa = cohen_kappa_score(y_true_numeric, y_pred_numeric)

    # Matthews Correlation Coefficient (for multiclass)
    mcc = matthews_corrcoef(y_true_numeric, y_pred_numeric)

    # Micro-averaged ROC-AUC
    fpr_micro, tpr_micro, _ = roc_curve(y_true_bin.ravel(), y_pred_proba.ravel())
    roc_auc_micro = auc(fpr_micro, tpr_micro)

    # Macro-averaged ROC-AUC
    roc_auc_macro = np.mean([per_class_metrics[cls]['roc_auc']
                             for cls in class_names])

    # Macro-averaged PR-AUC
    pr_auc_macro = np.mean([per_class_metrics[cls]['pr_auc']
                           for cls in class_names])

    print("\nOverall Performance Metrics:")
    print("-"*70)
    print(f"  Balanced Accuracy:     {balanced_acc:.4f}")
    print(f"  Cohen's Kappa:         {kappa:.4f}")
    print(f"  Matthews Corr. Coef:   {mcc:.4f}")
    print(f"\n  Macro-averaged:")
    print(f"    Precision:           {precision_macro:.4f}")
    print(f"    Recall:              {recall_macro:.4f}")
    print(f"    F1-Score:            {f1_macro:.4f}")
    print(f"    ROC-AUC:             {roc_auc_macro:.4f}")
    print(f"    PR-AUC:              {pr_auc_macro:.4f}")
    print(f"\n  Weighted-averaged:")
    print(f"    Precision:           {precision_weighted:.4f}")
    print(f"    Recall:              {recall_weighted:.4f}")
    print(f"    F1-Score:            {f1_weighted:.4f}")
    print(f"\n  Micro-averaged:")
    print(f"    ROC-AUC:             {roc_auc_micro:.4f}")

    print("\nPer-Class Performance:")
    print("-"*70)
    for idx, class_name in enumerate(class_names):
        print(f"\n  {class_name}:")
        print(f"    Precision:           {precision_per_class[idx]:.4f}")
        print(f"    Recall:              {recall_per_class[idx]:.4f}")
        print(f"    F1-Score:            {f1_per_class[idx]:.4f}")

    # ========================================
    # Create Comprehensive Visualization
    # ========================================

    print("\n" + "-"*70)
    print("Creating visualization figures...")

    # Figure 1: ROC Curves (One-vs-Rest)
    fig1, axes1 = plt.subplots(1, 2, figsize=(16, 6))

    # Left plot: Individual ROC curves
    ax1 = axes1[0]
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for idx, (class_name, color) in enumerate(zip(class_names, colors)):
        metrics = per_class_metrics[class_name]
        ax1.plot(metrics['fpr'], metrics['tpr'], color=color, lw=2.5,
                label=f"{class_labels[idx]} (AUC = {metrics['roc_auc']:.3f})")

    # Micro-average
    ax1.plot(fpr_micro, tpr_micro, color='deeppink', linestyle=':', lw=2.5,
            label=f'Micro-avg (AUC = {roc_auc_micro:.3f})')

    # Macro-average (interpolated)
    all_fpr = np.unique(np.concatenate([per_class_metrics[cls]['fpr']
                                        for cls in class_names]))
    mean_tpr = np.zeros_like(all_fpr)
    for class_name in class_names:
        metrics = per_class_metrics[class_name]
        mean_tpr += np.interp(all_fpr, metrics['fpr'], metrics['tpr'])
    mean_tpr /= n_classes
    fpr_macro = all_fpr
    tpr_macro = mean_tpr
    roc_auc_macro_interp = auc(fpr_macro, tpr_macro)

    ax1.plot(fpr_macro, tpr_macro, color='navy', linestyle='--', lw=2.5,
            label=f'Macro-avg (AUC = {roc_auc_macro_interp:.3f})')

    # Diagonal line
    ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5, label='Random (AUC = 0.500)')

    ax1.set_xlim([0.0, 1.0])
    ax1.set_ylim([0.0, 1.05])
    ax1.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
    ax1.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
    ax1.set_title('ROC Curves (One-vs-Rest)', fontsize=14, fontweight='bold')
    ax1.legend(loc="lower right", fontsize=10, framealpha=0.9)
    ax1.grid(True, alpha=0.3)

    # Right plot: PR Curves
    ax2 = axes1[1]

    for idx, (class_name, color) in enumerate(zip(class_names, colors)):
        metrics = per_class_metrics[class_name]
        ax2.plot(metrics['recall_curve'], metrics['precision_curve'],
                color=color, lw=2.5,
                label=f"{class_labels[idx]} (AP = {metrics['avg_precision']:.3f})")

    # Macro-average PR
    ax2.axhline(y=pr_auc_macro, color='navy', linestyle='--', lw=2.5,
               label=f'Macro-avg (AP = {pr_auc_macro:.3f})')

    ax2.set_xlim([0.0, 1.0])
    ax2.set_ylim([0.0, 1.05])
    ax2.set_xlabel('Recall', fontsize=13, fontweight='bold')
    ax2.set_ylabel('Precision', fontsize=13, fontweight='bold')
    ax2.set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
    ax2.legend(loc="best", fontsize=10, framealpha=0.9)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    fig1_path = f"{output_dir}/roc_pr_curves.png"
    plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"  âœ“ Saved: {fig1_path}")

    # Figure 2: Performance Metrics Comparison
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 12))

    # Plot 1: Per-class metrics bar chart
    ax1 = axes2[0, 0]
    x = np.arange(n_classes)
    width = 0.25

    bars1 = ax1.bar(x - width, precision_per_class, width, label='Precision',
                   color='#3498db', alpha=0.8, edgecolor='black')
    bars2 = ax1.bar(x, recall_per_class, width, label='Recall',
                   color='#e74c3c', alpha=0.8, edgecolor='black')
    bars3 = ax1.bar(x + width, f1_per_class, width, label='F1-Score',
                   color='#2ecc71', alpha=0.8, edgecolor='black')

    ax1.set_xlabel('Activity Class', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax1.set_title('Per-Class Performance Metrics', fontsize=13, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(class_labels, fontsize=10)
    ax1.legend(fontsize=11)
    ax1.grid(axis='y', alpha=0.3)
    ax1.set_ylim([0, 1.05])

    # Add value labels
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.3f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

    # Plot 2: Macro vs Weighted metrics
    ax2 = axes2[0, 1]
    metrics_comparison = {
        'Precision': [precision_macro, precision_weighted],
        'Recall': [recall_macro, recall_weighted],
        'F1-Score': [f1_macro, f1_weighted]
    }

    x2 = np.arange(len(metrics_comparison))
    width2 = 0.35

    macro_values = [v[0] for v in metrics_comparison.values()]
    weighted_values = [v[1] for v in metrics_comparison.values()]

    bars1 = ax2.bar(x2 - width2/2, macro_values, width2, label='Macro-avg',
                   color='#9b59b6', alpha=0.8, edgecolor='black')
    bars2 = ax2.bar(x2 + width2/2, weighted_values, width2, label='Weighted-avg',
                   color='#f39c12', alpha=0.8, edgecolor='black')

    ax2.set_xlabel('Metric', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax2.set_title('Macro vs Weighted Averaging', fontsize=13, fontweight='bold')
    ax2.set_xticks(x2)
    ax2.set_xticklabels(metrics_comparison.keys(), fontsize=11)
    ax2.legend(fontsize=11)
    ax2.grid(axis='y', alpha=0.3)
    ax2.set_ylim([0, 1.05])

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.3f}', ha='center', va='bottom',
                    fontsize=9, fontweight='bold')

    # Plot 3: ROC-AUC comparison
    ax3 = axes2[1, 0]
    roc_aucs = [per_class_metrics[cls]['roc_auc'] for cls in class_names]
    roc_aucs.extend([roc_auc_macro, roc_auc_micro])

    labels_roc = class_labels + ['Macro-avg', 'Micro-avg']
    colors_roc = colors + ['navy', 'deeppink']

    bars = ax3.barh(labels_roc, roc_aucs, color=colors_roc,
                    alpha=0.8, edgecolor='black')
    ax3.set_xlabel('ROC-AUC Score', fontsize=12, fontweight='bold')
    ax3.set_title('ROC-AUC Comparison', fontsize=13, fontweight='bold')
    ax3.grid(axis='x', alpha=0.3)
    ax3.set_xlim([0, 1.05])

    # Add value labels
    for bar, value in zip(bars, roc_aucs):
        ax3.text(value + 0.02, bar.get_y() + bar.get_height()/2.,
                f'{value:.3f}', va='center', fontweight='bold', fontsize=10)

    # Plot 4: Overall metrics summary
    ax4 = axes2[1, 1]
    overall_metrics = {
        'Balanced\nAccuracy': balanced_acc,
        "Cohen's\nKappa": kappa,
        'Matthews\nCorr. Coef': mcc,
        'Macro\nF1-Score': f1_macro,
        'Weighted\nF1-Score': f1_weighted
    }

    bars = ax4.bar(overall_metrics.keys(), overall_metrics.values(),
                   color=['#16a085', '#27ae60', '#2980b9', '#8e44ad', '#c0392b'],
                   alpha=0.8, edgecolor='black')
    ax4.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax4.set_title('Overall Classification Metrics', fontsize=13, fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)
    ax4.set_ylim([0, 1.05])
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')

    plt.tight_layout()
    fig2_path = f"{output_dir}/classification_metrics_summary.png"
    plt.savefig(fig2_path, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"  âœ“ Saved: {fig2_path}")

    # ========================================
    # Save Detailed Metrics to CSV
    # ========================================

    print("\n" + "-"*70)
    print("Saving detailed metrics to CSV files...")

    # Per-class metrics
    per_class_df = pd.DataFrame({
        'Class': class_names,
        'Precision': precision_per_class,
        'Recall': recall_per_class,
        'F1_Score': f1_per_class,
        'ROC_AUC': [per_class_metrics[cls]['roc_auc'] for cls in class_names],
        'PR_AUC': [per_class_metrics[cls]['pr_auc'] for cls in class_names],
        'Avg_Precision': [per_class_metrics[cls]['avg_precision'] for cls in class_names]
    })
    per_class_df.to_csv(f"{output_dir}/per_class_metrics.csv", index=False)
    print(f"  âœ“ Saved: per_class_metrics.csv")

    # Overall metrics
    overall_df = pd.DataFrame({
        'Metric': ['Balanced_Accuracy', 'Cohen_Kappa', 'Matthews_Corr_Coef',
                   'Precision_Macro', 'Recall_Macro', 'F1_Macro',
                   'Precision_Weighted', 'Recall_Weighted', 'F1_Weighted',
                   'ROC_AUC_Macro', 'ROC_AUC_Micro', 'PR_AUC_Macro'],
        'Score': [balanced_acc, kappa, mcc,
                  precision_macro, recall_macro, f1_macro,
                  precision_weighted, recall_weighted, f1_weighted,
                  roc_auc_macro, roc_auc_micro, pr_auc_macro]
    })
    overall_df.to_csv(f"{output_dir}/overall_classification_metrics.csv", index=False)
    print(f"  âœ“ Saved: overall_classification_metrics.csv")

    # Compile results dictionary
    metrics_dict = {
        'per_class': per_class_metrics,
        'overall': {
            'balanced_accuracy': balanced_acc,
            'cohen_kappa': kappa,
            'matthews_corrcoef': mcc,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'f1_macro': f1_macro,
            'precision_weighted': precision_weighted,
            'recall_weighted': recall_weighted,
            'f1_weighted': f1_weighted,
            'roc_auc_macro': roc_auc_macro,
            'roc_auc_micro': roc_auc_micro,
            'pr_auc_macro': pr_auc_macro
        },
        'per_class_arrays': {
            'precision': precision_per_class,
            'recall': recall_per_class,
            'f1': f1_per_class
        }
    }

    print("\n" + "="*70)
    print("âœ“ Advanced classification metrics computation completed!")
    print("="*70)

    return metrics_dict

In [ ]:
#==============================================================================
# 18. Main Execution
#==============================================================================

if __name__ == "__main__":
    print("="*70)
    print("GSK3-Beta QSAR Modeling Pipeline")
    print("="*70)
    print("Enhanced Version with:")
    print("  1. ProLIF-Validated GSK3 Descriptors (8 features, p<0.05)")
    print("  2. mRMR Feature Selection (with ProLIF protection)")
    print("  3. Ablation Study (feature group importance)")
    print("  4. Feature Number Optimization (100-600 range)")
    print("  5. Bayesian-Optimized Ensemble")
    print("  6. SHAP Analysis")
    print("  7. Automated Flowchart Generation")
    print("  8. Publication-Ready Visualizations")
    print("\nProLIF-Validated GSK3 Descriptors:")
    print("  - GSK002: HBond_Donors_Ratio (p=0.0419)")
    print("  - GSK004: Total_HBond_Potential (p=0.0180)")
    print("  - GSK007: Aromatic_Carbocycles (p=0.0237)")
    print("  - GSK008: Aromatic_Heterocycles (p=0.0074)")
    print("  - GSK015: Rotatable_Bonds (p=0.0101)")
    print("  - GSK026: Negative_Charges (p=0.0101)")
    print("  - GSK037: Basic_N_Ratio (p=0.0306)")
    print("  - GSK043: Aromatic_Ring_Fraction (p=0.0074)")
    print("="*70)

    input_csv_file = 'GSK3B_in_vitro.csv'

    if os.path.exists(input_csv_file):
        try:
            # ========================================
            # STEP 1: Run Main Pipeline
            # ========================================
            print("\n" + "="*70)
            print("STEP 1: RUNNING MAIN QSAR PIPELINE")
            print("="*70)

            pipeline_results = run_gsk3_pipeline(input_csv_file)
            print("\nâœ“ Pipeline execution completed successfully!")

            # ========================================
            # STEP 2: Load Test Results for Visualization
            # ========================================
            print("\n" + "="*70)
            print("STEP 2: LOADING TEST RESULTS")
            print("="*70)

            output_dir = pipeline_results['output_directory']
            results_df = pd.read_csv(f"{output_dir}/test_predictions.csv")
            y_test = results_df['Actual_pChEMBL'].values
            y_pred = results_df['Predicted_pChEMBL'].values

            print(f"âœ“ Loaded {len(y_test)} test predictions")

            # ========================================
            # STEP 3: Generate Publication Figures
            # ========================================
            print("\n" + "="*70)
            print("STEP 3: GENERATING PUBLICATION-READY FIGURES")
            print("="*70)

            figure_paths = generate_all_publication_figures(
                pipeline_results, y_test, y_pred
            )

            # ========================================
            # STEP 4: Generate Methodology Flowcharts
            # ========================================
            print("\n" + "="*70)
            print("STEP 4: GENERATING METHODOLOGY FLOWCHARTS")
            print("="*70)

            try:
                # Generate standard flowcharts
                flowchart_dir = generate_methodology_flowcharts_automated(pipeline_results)

                # Generate feature transformation flowchart
                feature_flow_path = generate_feature_transformation_flowchart(pipeline_results)

                print("\nâœ“ All flowcharts generated successfully!")
                print(f"\nFlowcharts saved to: {flowchart_dir}/")
                print("  1. 01_main_pipeline_automated.png")
                print("  2. 02_feature_engineering_automated.png")
                print("  3. 03_ensemble_architecture_automated.png")
                print("  4. 04_feature_transformation_detailed.png")

            except Exception as e:
                print(f"\nâœ— Flowchart generation failed: {e}")
                print("Note: Install graphviz if needed: pip install graphviz")

            # ========================================
            # STEP 5: Generate Summary Report
            # ========================================
            print("\n" + "="*70)
            print("STEP 5: GENERATING SUMMARY REPORT")
            print("="*70)

            summary_file = f"{output_dir}/publication_summary.txt"
            with open(summary_file, 'w') as f:
                f.write("="*70 + "\n")
                f.write("GSK3-Î² QSAR MODEL - PUBLICATION SUMMARY\n")
                f.write("="*70 + "\n\n")

                f.write("MODEL PERFORMANCE\n")
                f.write("-"*70 + "\n")
                f.write(f"RÂ² Score:  {pipeline_results['performance']['r2']:.4f}\n")
                f.write(f"RMSE:      {pipeline_results['performance']['rmse']:.4f}\n")
                f.write(f"MAE:       {pipeline_results['performance']['mae']:.4f}\n\n")

                stats = pipeline_results['pipeline_statistics']

                f.write("DATASET STATISTICS\n")
                f.write("-"*70 + "\n")
                f.write(f"Total compounds:     {stats['preprocessing']['final_compounds']:,}\n")
                f.write(f"Training samples:    {stats['training']['train_samples']:,}\n")
                f.write(f"Validation samples:  {stats['training']['validation_samples']:,}\n")
                f.write(f"Test samples:        {stats['training']['test_samples']:,}\n\n")

                f.write("FEATURE ENGINEERING\n")
                f.write("-"*70 + "\n")
                f.write(f"Initial features:         {stats['features']['total_features']:,}\n")
                f.write(f"After preprocessing:      {stats['features']['processed_features']:,}\n")
                if 'feature_selection' in stats:
                    f.write(f"After optimization:       {stats['feature_selection']['n_selected']}\n")
                    f.write(f"ProLIF protected:         {stats['feature_selection']['prolif_protected']}\n")
                    reduction_rate = (1-stats['feature_selection']['n_selected']/stats['features']['total_features'])*100
                    f.write(f"Reduction rate:           {reduction_rate:.1f}%\n\n")

                f.write("GENERATED FIGURES\n")
                f.write("-"*70 + "\n")
                for name, path in figure_paths.items():
                    if path:
                        f.write(f"  âœ“ {name}\n")
                f.write("\n")

                f.write("ENSEMBLE MODEL WEIGHTS\n")
                f.write("-"*70 + "\n")
                for model, weight in stats['models']['ensemble_weights'].items():
                    f.write(f"  {model:20s}: {weight*100:5.1f}%\n")

                f.write("\n" + "="*70 + "\n")
                f.write("All results and figures are publication-ready\n")
                f.write("Verifiable through pipeline_statistics.json\n")
                f.write("="*70 + "\n")

            print(f"âœ“ Summary report saved to {summary_file}")

            # ========================================
            # FINAL SUCCESS MESSAGE
            # ========================================
            print("\n" + "="*70)
            print("PIPELINE COMPLETED SUCCESSFULLY!")
            print("="*70)
            print(f"\nAll results saved to: {output_dir}/")
            print("\nGenerated files:")
            print("  Statistical Results:")
            print("     - pipeline_statistics.json")
            print("     - performance_summary.txt")
            print("     - publication_summary.txt")
            print("\n  Performance Figures:")
            print("     - prediction_scatter.png")
            print("     - feature_importance.png")
            print("     - activity_distribution_comparison.png")
            print("     - activity_confusion_matrix.png")
            print("     - residual_analysis.png")
            print("     - model_performance_comparison.png")
            print("     - feature_group_contribution.png")
            print("\n  Flowcharts:")
            print("     - 01_main_pipeline_automated.png")
            print("     - 02_feature_engineering_automated.png")
            print("     - 03_ensemble_architecture_automated.png")
            print("     - 04_feature_transformation_detailed.png")
            print("\n  Data Files:")
            print("     - test_predictions.csv")
            print("     - feature_importance.csv")
            print("     - activity_distribution_stats.csv")
            print("     - activity_classification_report.csv")
            print("\nâœ“ All figures are publication-ready")
            print("âœ“ All statistics are verifiable from pipeline_statistics.json")
            print("âœ“ Ready for thesis/paper submission")
            print("="*70 + "\n")

        except Exception as e:
            print(f"\nâœ— Error during pipeline execution: {e}")
            import traceback
            traceback.print_exc()

    else:
        print(f"\nâœ— Error: Input file '{input_csv_file}' not found!")
        print("Please ensure GSK3B_in_vitro.csv is in the current directory.")

GSK3-Beta QSAR Modeling Pipeline
Enhanced Version with:
  1. ProLIF-Validated GSK3 Descriptors (8 features, p<0.05)
  2. mRMR Feature Selection (with ProLIF protection)
  3. Ablation Study (feature group importance)
  4. Feature Number Optimization (100-600 range)
  5. Bayesian-Optimized Ensemble
  6. SHAP Analysis
  7. Automated Flowchart Generation
  8. Publication-Ready Visualizations

ProLIF-Validated GSK3 Descriptors:
  - GSK002: HBond_Donors_Ratio (p=0.0419)
  - GSK004: Total_HBond_Potential (p=0.0180)
  - GSK007: Aromatic_Carbocycles (p=0.0237)
  - GSK008: Aromatic_Heterocycles (p=0.0074)
  - GSK015: Rotatable_Bonds (p=0.0101)
  - GSK026: Negative_Charges (p=0.0101)
  - GSK037: Basic_N_Ratio (p=0.0306)
  - GSK043: Aromatic_Ring_Fraction (p=0.0074)

STEP 1: RUNNING MAIN QSAR PIPELINE

GSK3-Beta QSAR MODELING PIPELINE
ENHANCED VERSION with:
  1. ProLIF-Validated GSK3 Descriptors (8 features, p<0.05)
  2. mRMR Feature Selection (with ProLIF protection)
  3. Ablation Study (feature 